# ARC-AGI-3 LLM Agent

LLM-based agent using Qwen3.6-27B-FP8 via vLLM. 
Architecture: tool-calling REPL with Python sandbox, anti-loop detection, 
context management. Based on the Duck harness (MIT, Tufa Labs).

**Datasets required:**
- `driessmit1/arc3-vllm-h100-wheelhouse-v3` (vLLM + torch wheels)
- `driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot` (Qwen3.6-27B-FP8 model)


In [ ]:
import base64
import json
import os
import shutil
import subprocess
import sys
import sysconfig
import time
from pathlib import Path

# Competition rerun detection
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
NOTEBOOK_START = time.time()

os.environ["MPLBACKEND"] = "Agg"

# CUDA library path
cuda_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    e for e in [cuda_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if e
)

WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

print(f"TRUE_SUBMISSION={TRUE_SUBMISSION}")
print(f"GPU check:")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=False)


In [ ]:
# Install ARC runtime from competition wheels
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "--no-index", "--no-warn-conflicts", "--disable-pip-version-check",
    "--find-links", "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
    "arc-agi", "python-dotenv",
], stdout=subprocess.DEVNULL)
print("ARC runtime installed")


In [ ]:
# Install vLLM from the wheelhouse dataset
WHEELHOUSE_DATASET = "driessmit1/arc3-vllm-h100-wheelhouse-v3"
MODEL_DATASET = "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"

# Find the wheelhouse and model paths
def find_dataset_path(slug):
    for candidate in [
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/datasets") / slug.split("/")[0] / slug.split("/")[1],
    ]:
        if candidate.exists():
            return candidate
    # Fallback: search for the slug directory
    for p in Path("/kaggle/input").iterdir():
        if slug.split("/")[-1] in p.name or p.name == slug.split("/")[-1]:
            return p
    return Path(f"/kaggle/input/{slug.split('/')[-1]}")

WHEELHOUSE = find_dataset_path(WHEELHOUSE_DATASET)
MODEL_PATH = find_dataset_path(MODEL_DATASET)

print(f"Wheelhouse: {WHEELHOUSE} (exists: {WHEELHOUSE.exists()})")
print(f"Model: {MODEL_PATH} (exists: {MODEL_PATH.exists()})")

# Install vLLM into a separate site-packages to avoid conflicts
SITE_PACKAGES = WORKING / "vllm_site"
SITE_PACKAGES.mkdir(exist_ok=True)

requirements = WHEELHOUSE / "requirements.lock"
if not requirements.exists():
    # Try alternative names
    for name in ["requirements.txt", "requirements.lock"]:
        alt = WHEELHOUSE / name
        if alt.exists():
            requirements = alt
            break

if requirements.exists():
    print(f"Installing vLLM from {requirements}")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--no-index", "--find-links", str(WHEELHOUSE),
        "--requirement", str(requirements),
        "--target", str(SITE_PACKAGES),
        "--upgrade", "--ignore-installed",
        "--only-binary", ":all:", "--no-compile",
        "--disable-pip-version-check", "--no-warn-conflicts",
    ])
    print("vLLM installed")
else:
    print(f"WARNING: No requirements file found in {WHEELHOUSE}")
    print(f"Contents: {list(WHEELHOUSE.iterdir())[:10]}")
    # Try direct install of vllm wheel
    wheels = list(WHEELHOUSE.glob("vllm-*.whl"))
    if wheels:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "--no-index", "--find-links", str(WHEELHOUSE),
            "--target", str(SITE_PACKAGES),
            str(wheels[0]),
        ])
        print(f"Installed vLLM wheel: {wheels[0].name}")

# Add to Python path
os.environ["PYTHONPATH"] = str(SITE_PACKAGES) + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, str(SITE_PACKAGES))


In [ ]:
import urllib.request

VLLM_HOST = "127.0.0.1"
VLLM_PORT = 1234
VLLM_BASE_URL = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"
SERVED_MODEL = "vrfai/Qwen3.6-27B-FP8"
VLLM_LOG = WORKING / "vllm_server.log"
VLLM_PID_FILE = WORKING / "vllm_server.pid"

# Kill any existing server
if VLLM_PID_FILE.exists():
    try:
        os.kill(int(VLLM_PID_FILE.read_text().strip()), 9)
    except Exception:
        pass
    VLLM_PID_FILE.unlink(missing_ok=True)

# Start vLLM OpenAI-compatible server
vllm_env = os.environ.copy()
vllm_env.update({
    "USE_TF": "0",
    "TRANSFORMERS_NO_TF": "1",
    "TRANSFORMERS_NO_TORCHVISION": "1",
    "VLLM_NO_USAGE_STATS": "1",
    "PYTHONPATH": str(SITE_PACKAGES),
})

cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", str(MODEL_PATH),
    "--served-model-name", SERVED_MODEL,
    "--host", VLLM_HOST,
    "--port", str(VLLM_PORT),
    "--tensor-parallel-size", "1",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_coder",
    "--generation-config", "vllm",
    "--enable-prefix-caching",
    "--default-chat-template-kwargs", '{"preserve_thinking": true}',
    "--reasoning-parser", "qwen3",
    "--max-model-len", "65536",
]

print(f"Starting vLLM: {' '.join(cmd[:8])}...")
log_fh = open(VLLM_LOG, "w")
proc = subprocess.Popen(cmd, env=vllm_env, stdout=log_fh, stderr=subprocess.STDOUT, text=True)
VLLM_PID_FILE.write_text(str(proc.pid))

# Wait for server to be ready
print("Waiting for vLLM server...", flush=True)
deadline = time.time() + 600
ready = False
while time.time() < deadline:
    try:
        req = urllib.request.Request(f"{VLLM_BASE_URL}/models")
        with urllib.request.urlopen(req, timeout=5) as resp:
            if resp.status < 500:
                ready = True
                break
    except Exception:
        pass
    time.sleep(5)
    # Check if process is still alive
    if proc.poll() is not None:
        print(f"vLLM server exited with code {proc.returncode}")
        print(f"Log tail: {VLLM_LOG.read_text()[-2000:]}")
        break

if ready:
    print("vLLM server ready!")
    # Smoke test
    payload = json.dumps({
        "model": SERVED_MODEL,
        "messages": [{"role": "user", "content": "Say hello in one word."}],
        "max_tokens": 16,
        "temperature": 0.0,
    }).encode()
    req = urllib.request.Request(
        f"{VLLM_BASE_URL}/chat/completions",
        data=payload,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        result = json.loads(resp.read())
    reply = result["choices"][0]["message"].get("content", "").strip()
    print(f"Smoke test reply: {reply!r}")
    VLLM_OK = True
else:
    print("ERROR: vLLM server did not start. Log:")
    print(VLLM_LOG.read_text()[-3000:])
    VLLM_OK = False


In [ ]:

# Deploy agent source files
agent_dir = Path("/tmp/arc_llm_agent")
agent_dir.mkdir(parents=True, exist_ok=True)

(agent_dir / "__init__.py").write_text("""# LLM agent package""")
(agent_dir / "grid_utils.py").write_bytes(base64.b64decode("IiIiR3JpZCBmb3JtYXR0aW5nIGFuZCBzZWdtZW50YXRpb24gdXRpbGl0aWVzIGZvciBBUkMtQUdJLTMgTExNIGFnZW50LgoKRGVyaXZlZCBmcm9tIHRoZSBEdWNrIGhhcm5lc3MgKE1JVCBsaWNlbnNlLCBUdWZhIExhYnMpLgpTaW1wbGlmaWVkIGZvciB1c2Ugd2l0aCB0aGUgQVJDLUFHSS0zLUFnZW50cyBmcmFtZXdvcmsuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBTZXF1ZW5jZQoKIyBBUkMgY29sb3IgcGFsZXR0ZTogaW5kZXggLT4gc2luZ2xlLWNoYXIgbGFiZWwKQVJDX0NPTE9SX0NIQVJTID0gIld3Z0djQk1QUmJTWU9yTnAiCkFSQ19DT0xPUl9MRUdFTkQgPSAoCiAgICAiVz13aGl0ZSwgdz1saWdodCBncmF5LCBnPWdyYXksIEc9ZGFyayBncmF5LCBjPWNoYXJjb2FsLCBCPWJsYWNrLCAiCiAgICAiTT1tYWdlbnRhLCBQPXBpbmssIFI9cmVkLCBiPWJsdWUsIFM9c2t5IGJsdWUsIFk9eWVsbG93LCBPPW9yYW5nZSwgIgogICAgInI9ZGFyayByZWQsIE49bGlnaHQgZ3JlZW4sIHA9cHVycGxlIgopCgojIEFSQyBjb2xvciBwYWxldHRlOiBpbmRleCAtPiBSR0IKQVJDX0NPTE9SX01BUDogZGljdFtpbnQsIHR1cGxlW2ludCwgaW50LCBpbnRdXSA9IHsKICAgIDA6ICgyNTUsIDI1NSwgMjU1KSwgMTogKDIwNCwgMjA0LCAyMDQpLCAyOiAoMTUzLCAxNTMsIDE1MyksCiAgICAzOiAoMTAyLCAxMDIsIDEwMiksIDQ6ICg1MSwgNTEsIDUxKSwgICAgNTogKDAsIDAsIDApLAogICAgNjogKDIyOSwgNTgsIDE2MyksICA3OiAoMjU1LCAxMjMsIDIwNCksIDg6ICgyNDksIDYwLCA0OSksCiAgICA5OiAoMzAsIDE0NywgMjU1KSwgIDEwOiAoMTM2LCAyMTYsIDI0MSksIDExOiAoMjU1LCAyMjAsIDApLAogICAgMTI6ICgyNTUsIDEzMywgMjcpLCAxMzogKDE0NiwgMTgsIDQ5KSwgIDE0OiAoNzksIDIwNCwgNDgpLAogICAgMTU6ICgxNjMsIDg2LCAyMTQpLAp9CgoKZGVmIGZvcm1hdF9ncmlkX2FzY2lpKGdyaWQ6IFNlcXVlbmNlW1NlcXVlbmNlW2ludF1dKSAtPiBzdHI6CiAgICAiIiJSZW5kZXIgYSBudW1lcmljIGdyaWQgYXMgQVNDSUkgYXJ0IHVzaW5nIEFSQyBjb2xvciBjaGFyYWN0ZXJzLiIiIgogICAgaWYgbm90IGdyaWQ6CiAgICAgICAgcmV0dXJuICIoZW1wdHkgZ3JpZCkiCiAgICBsaW5lcyA9IFtdCiAgICBmb3Igcm93IGluIGdyaWQ6CiAgICAgICAgY2hhcnMgPSBbXQogICAgICAgIGZvciB2IGluIHJvdzoKICAgICAgICAgICAgY2hhcnMuYXBwZW5kKEFSQ19DT0xPUl9DSEFSU1ttYXgoMCwgbWluKDE1LCBpbnQodikpKV0pCiAgICAgICAgbGluZXMuYXBwZW5kKCIiLmpvaW4oY2hhcnMpKQogICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykKCgpkZWYgZ3JpZF90b19wbmdfYnl0ZXMoZ3JpZDogU2VxdWVuY2VbU2VxdWVuY2VbaW50XV0sIHVwc2NhbGU6IGludCA9IDQpIC0+IGJ5dGVzOgogICAgIiIiUmVuZGVyIGEgbnVtZXJpYyBncmlkIGFzIGEgUE5HIGltYWdlIChubyBQSUwgZGVwZW5kZW5jeSDigJQgcHVyZSBieXRlcykuIiIiCiAgICB0cnk6CiAgICAgICAgZnJvbSBQSUwgaW1wb3J0IEltYWdlCiAgICAgICAgaW1wb3J0IGlvCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgcmV0dXJuIGIiIgogICAgCiAgICByb3dzID0gbGVuKGdyaWQpCiAgICBjb2xzID0gbWF4KChsZW4ocikgZm9yIHIgaW4gZ3JpZCksIGRlZmF1bHQ9MCkgaWYgcm93cyBlbHNlIDAKICAgIGlmIHJvd3MgPD0gMCBvciBjb2xzIDw9IDA6CiAgICAgICAgcmV0dXJuIGIiIgogICAgCiAgICBpbWFnZSA9IEltYWdlLm5ldygiUkdCIiwgKGNvbHMsIHJvd3MpLCBBUkNfQ09MT1JfTUFQWzBdKQogICAgcGl4ZWxzID0gaW1hZ2UubG9hZCgpCiAgICBmb3Igcm93X2lkeCwgcm93IGluIGVudW1lcmF0ZShncmlkKToKICAgICAgICBmb3IgY29sX2lkeCBpbiByYW5nZShjb2xzKToKICAgICAgICAgICAgdmFsdWUgPSByb3dbY29sX2lkeF0gaWYgY29sX2lkeCA8IGxlbihyb3cpIGVsc2UgMAogICAgICAgICAgICBwaXhlbHNbY29sX2lkeCwgcm93X2lkeF0gPSBBUkNfQ09MT1JfTUFQLmdldChpbnQodmFsdWUpLCBBUkNfQ09MT1JfTUFQWzBdKQogICAgCiAgICBpZiB1cHNjYWxlID4gMToKICAgICAgICBpbWFnZSA9IGltYWdlLnJlc2l6ZSgoY29scyAqIHVwc2NhbGUsIHJvd3MgKiB1cHNjYWxlKSwgSW1hZ2UuUmVzYW1wbGluZy5ORUFSRVNUKQogICAgCiAgICBidWZmZXIgPSBpby5CeXRlc0lPKCkKICAgIGltYWdlLnNhdmUoYnVmZmVyLCBmb3JtYXQ9IlBORyIpCiAgICByZXR1cm4gYnVmZmVyLmdldHZhbHVlKCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIFNlZ21lbnRhdGlvbiAoY29ubmVjdGVkLWNvbXBvbmVudCBhbmFseXNpcykKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCl9PUlRIID0gKCgtMSwgMCksICgxLCAwKSwgKDAsIC0xKSwgKDAsIDEpKQpfQ1cgPSAoKC0xLCAtMSksICgtMSwgMCksICgtMSwgMSksICgwLCAxKSwgKDEsIDEpLCAoMSwgMCksICgxLCAtMSksICgwLCAtMSkpCl9DV19JTkRFWCA9IHtvZmY6IGkgZm9yIGksIG9mZiBpbiBlbnVtZXJhdGUoX0NXKX0KCgpkZWYgX3RyYWNlX291dGVyX2NvbnRvdXIoY2VsbHMsIHN0YXJ0KToKICAgICIiIk1vb3JlLW5laWdoYm91ciB0cmFjZSBvZiBvdXRlciBwZXJpbWV0ZXIsIGNsb2Nrd2lzZS4iIiIKICAgIGlmIGxlbihjZWxscykgPT0gMToKICAgICAgICByZXR1cm4gW3N0YXJ0XQogICAgY29udG91ciA9IFtzdGFydF0KICAgIGIgPSBzdGFydAogICAgcHJldiA9IChzdGFydFswXSwgc3RhcnRbMV0gLSAxKQogICAgc2Vjb25kID0gTm9uZQogICAgZm9yIF8gaW4gcmFuZ2UoOCAqIGxlbihjZWxscykgKyAxNik6CiAgICAgICAgaWR4ID0gX0NXX0lOREVYWyhwcmV2WzBdIC0gYlswXSwgcHJldlsxXSAtIGJbMV0pXQogICAgICAgIG54dCA9IE5vbmUKICAgICAgICBmb3IgayBpbiByYW5nZSgxLCA5KToKICAgICAgICAgICAgb2ZmID0gX0NXWyhpZHggKyBrKSAlIDhdCiAgICAgICAgICAgIGNhbmQgPSAoYlswXSArIG9mZlswXSwgYlsxXSArIG9mZlsxXSkKICAgICAgICAgICAgaWYgY2FuZCBpbiBjZWxsczoKICAgICAgICAgICAgICAgIG54dCA9IGNhbmQKICAgICAgICAgICAgICAgIGJhY2sgPSBfQ1dbKGlkeCArIGsgLSAxKSAlIDhdCiAgICAgICAgICAgICAgICBuZXdfcHJldiA9IChiWzBdICsgYmFja1swXSwgYlsxXSArIGJhY2tbMV0pCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIGlmIG54dCBpcyBOb25lOgogICAgICAgICAgICBicmVhawogICAgICAgIGlmIHNlY29uZCBpcyBOb25lOgogICAgICAgICAgICBzZWNvbmQgPSBueHQKICAgICAgICBlbGlmIGIgPT0gc3RhcnQgYW5kIG54dCA9PSBzZWNvbmQ6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgY29udG91ci5hcHBlbmQobnh0KQogICAgICAgIHByZXYsIGIgPSBuZXdfcHJldiwgbnh0CiAgICBpZiBsZW4oY29udG91cikgPiAxIGFuZCBjb250b3VyWy0xXSA9PSBjb250b3VyWzBdOgogICAgICAgIGNvbnRvdXIucG9wKCkKICAgIHJldHVybiBjb250b3VyCgoKZGVmIF9jb3JuZXJfcG9pbnRzKGNvbnRvdXIpOgogICAgIiIiUmVkdWNlIGNvbnRvdXIgdG8gZGlyZWN0aW9uLWNoYW5nZSBwb2ludHMuIiIiCiAgICBpZiBsZW4oY29udG91cikgPD0gMjoKICAgICAgICByZXR1cm4gbGlzdChjb250b3VyKQogICAgbSA9IGxlbihjb250b3VyKQogICAgY29ybmVycyA9IFtdCiAgICBmb3IgaSBpbiByYW5nZShtKToKICAgICAgICBwcmV2X2MsIGN1ciwgbnh0ID0gY29udG91cltpIC0gMV0sIGNvbnRvdXJbaV0sIGNvbnRvdXJbKGkgKyAxKSAlIG1dCiAgICAgICAgZF9pbiA9IChjdXJbMF0gLSBwcmV2X2NbMF0sIGN1clsxXSAtIHByZXZfY1sxXSkKICAgICAgICBkX291dCA9IChueHRbMF0gLSBjdXJbMF0sIG54dFsxXSAtIGN1clsxXSkKICAgICAgICBpZiBkX2luICE9IGRfb3V0OgogICAgICAgICAgICBjb3JuZXJzLmFwcGVuZChjdXIpCiAgICByZXR1cm4gY29ybmVycwoKCmRlZiBfb2JqZWN0X2hhc2goY2VsbHMsIGNvbG9yKToKICAgICIiIlRyYW5zbGF0aW9uLWludmFyaWFudCBzaWduYXR1cmUgb2YgYW4gb2JqZWN0LiIiIgogICAgbWluX3IgPSBtaW4ociBmb3IgciwgXyBpbiBjZWxscykKICAgIG1pbl9jID0gbWluKGMgZm9yIF8sIGMgaW4gY2VsbHMpCiAgICBub3JtID0gc29ydGVkKChyIC0gbWluX3IsIGMgLSBtaW5fYykgZm9yIHIsIGMgaW4gY2VsbHMpCiAgICBwYXlsb2FkID0gcmVwcigoY29sb3IsIG5vcm0pKS5lbmNvZGUoKQogICAgcmV0dXJuIGhhc2hsaWIuc2hhMShwYXlsb2FkKS5oZXhkaWdlc3QoKVs6MTZdCgoKZGVmIHNlZ21lbnRfZ3JpZChncmlkOiBTZXF1ZW5jZVtTZXF1ZW5jZVtpbnRdXSkgLT4gZGljdFtzdHIsIEFueV06CiAgICAiIiJTZWdtZW50IGEgZ3JpZCBpbnRvIGNvbm5lY3RlZCBjb21wb25lbnRzLgogICAgCiAgICBSZXR1cm5zIGRpY3Qgd2l0aDoKICAgICAgLSBub2RlczogbGlzdCBvZiB7aWQsIGNvbG9yLCBoYXNoLCBwaXhlbHMsIGJvdW5kYXJ5LCBjaGlsZHJlbn0KICAgICAgLSBhZGphY2VuY3lfbGlzdDogbGlzdCBvZiBbaSwgal0gcGFpcnMKICAgICIiIgogICAgaGVpZ2h0ID0gbGVuKGdyaWQpCiAgICB3aWR0aCA9IGxlbihncmlkWzBdKSBpZiBoZWlnaHQgZWxzZSAwCiAgICBpZiBub3QgaGVpZ2h0IG9yIG5vdCB3aWR0aDoKICAgICAgICByZXR1cm4geyJub2RlcyI6IFtdLCAiYWRqYWNlbmN5X2xpc3QiOiBbXX0KICAgIAogICAgIyBDb25uZWN0ZWQgY29tcG9uZW50cyB2aWEgZmxvb2QgZmlsbAogICAgY29tcF9pZCA9IFtbLTFdICogd2lkdGggZm9yIF8gaW4gcmFuZ2UoaGVpZ2h0KV0KICAgIGNvbXBvbmVudHMgPSBbXQogICAgCiAgICBmb3Igc3IgaW4gcmFuZ2UoaGVpZ2h0KToKICAgICAgICBmb3Igc2MgaW4gcmFuZ2Uod2lkdGgpOgogICAgICAgICAgICBpZiBjb21wX2lkW3NyXVtzY10gIT0gLTE6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICB2YWx1ZSA9IGdyaWRbc3JdW3NjXQogICAgICAgICAgICBjaWQgPSBsZW4oY29tcG9uZW50cykKICAgICAgICAgICAgY2VsbHMgPSBzZXQoKQogICAgICAgICAgICBzdGFjayA9IFsoc3IsIHNjKV0KICAgICAgICAgICAgd2hpbGUgc3RhY2s6CiAgICAgICAgICAgICAgICByLCBjID0gc3RhY2sucG9wKCkKICAgICAgICAgICAgICAgIGlmIChyLCBjKSBpbiBjZWxsczoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgaWYgMCA8PSByIDwgaGVpZ2h0IGFuZCAwIDw9IGMgPCB3aWR0aCBhbmQgZ3JpZFtyXVtjXSA9PSB2YWx1ZSBhbmQgY29tcF9pZFtyXVtjXSA9PSAtMToKICAgICAgICAgICAgICAgICAgICBjZWxscy5hZGQoKHIsIGMpKQogICAgICAgICAgICAgICAgICAgIGNvbXBfaWRbcl1bY10gPSBjaWQKICAgICAgICAgICAgICAgICAgICBmb3IgZHIsIGRjIGluIF9PUlRIOgogICAgICAgICAgICAgICAgICAgICAgICBuciwgbmMgPSByICsgZHIsIGMgKyBkYwogICAgICAgICAgICAgICAgICAgICAgICBpZiAwIDw9IG5yIDwgaGVpZ2h0IGFuZCAwIDw9IG5jIDwgd2lkdGggYW5kIGNvbXBfaWRbbnJdW25jXSA9PSAtMSBhbmQgZ3JpZFtucl1bbmNdID09IHZhbHVlOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgc3RhY2suYXBwZW5kKChuciwgbmMpKQogICAgICAgICAgICBjb21wb25lbnRzLmFwcGVuZCh7InZhbHVlIjogdmFsdWUsICJjZWxscyI6IGNlbGxzLCAic3RhcnQiOiAoc3IsIHNjKX0pCiAgICAKICAgICMgQnVpbGQgbm9kZXMKICAgIG5vZGVzID0gW10KICAgIGZvciBjaWQsIGNvbXAgaW4gZW51bWVyYXRlKGNvbXBvbmVudHMpOgogICAgICAgIGNvbG9yX2NoYXIgPSBBUkNfQ09MT1JfQ0hBUlNbbWF4KDAsIG1pbigxNSwgY29tcFsidmFsdWUiXSkpXQogICAgICAgIG9ial9oYXNoID0gX29iamVjdF9oYXNoKGNvbXBbImNlbGxzIl0sIGNvbXBbInZhbHVlIl0pCiAgICAgICAgY29udG91ciA9IF90cmFjZV9vdXRlcl9jb250b3VyKGNvbXBbImNlbGxzIl0sIGNvbXBbInN0YXJ0Il0pCiAgICAgICAgYm91bmRhcnkgPSBfY29ybmVyX3BvaW50cyhjb250b3VyKQogICAgICAgIG5vZGVzLmFwcGVuZCh7CiAgICAgICAgICAgICJpZCI6IGNpZCwKICAgICAgICAgICAgImNvbG9yIjogY29sb3JfY2hhciwKICAgICAgICAgICAgImhhc2giOiBvYmpfaGFzaCwKICAgICAgICAgICAgInBpeGVscyI6IGxlbihjb21wWyJjZWxscyJdKSwKICAgICAgICAgICAgImJvdW5kYXJ5IjogW1tyLCBjXSBmb3IgciwgYyBpbiBib3VuZGFyeVs6MTJdXSwgICMgbGltaXQgc2l6ZQogICAgICAgICAgICAiY2hpbGRyZW4iOiBbXSwKICAgICAgICB9KQogICAgCiAgICAjIENvbXB1dGUgY29udGFpbm1lbnQgKGNoaWxkcmVuKQogICAgZm9yIG91dGVyX2lkLCBvdXRlcl9jb21wIGluIGVudW1lcmF0ZShjb21wb25lbnRzKToKICAgICAgICBvdXRlcl9jZWxscyA9IG91dGVyX2NvbXBbImNlbGxzIl0KICAgICAgICBmb3IgaW5uZXJfaWQsIGlubmVyX2NvbXAgaW4gZW51bWVyYXRlKGNvbXBvbmVudHMpOgogICAgICAgICAgICBpZiBpbm5lcl9pZCA9PSBvdXRlcl9pZDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICMgQ2hlY2sgaWYgaW5uZXIgaXMgZnVsbHkgZW5jbG9zZWQgYnkgb3V0ZXIKICAgICAgICAgICAgaW5uZXJfY2VsbHMgPSBpbm5lcl9jb21wWyJjZWxscyJdCiAgICAgICAgICAgIGlmIGFsbChhbnkoKHIgKyBkciwgYyArIGRjKSBpbiBvdXRlcl9jZWxscyBmb3IgZHIsIGRjIGluIF9PUlRIKQogICAgICAgICAgICAgICAgICAgZm9yIHIsIGMgaW4gaW5uZXJfY2VsbHMpOgogICAgICAgICAgICAgICAgIyBDaGVjayBubyBwYXRoIGZyb20gaW5uZXIgdG8gZ3JpZCBlZGdlIHdpdGhvdXQgY3Jvc3Npbmcgb3V0ZXIKICAgICAgICAgICAgICAgICMgU2ltcGxlIGNoZWNrOiBhbGwgbmVpZ2hib3JzIG9mIGlubmVyIGNlbGxzIGFyZSBlaXRoZXIgaW5uZXIgb3Igb3V0ZXIKICAgICAgICAgICAgICAgIGlzX2NoaWxkID0gVHJ1ZQogICAgICAgICAgICAgICAgZm9yIHIsIGMgaW4gaW5uZXJfY2VsbHM6CiAgICAgICAgICAgICAgICAgICAgZm9yIGRyLCBkYyBpbiBfT1JUSDoKICAgICAgICAgICAgICAgICAgICAgICAgbnIsIG5jID0gciArIGRyLCBjICsgZGMKICAgICAgICAgICAgICAgICAgICAgICAgaWYgMCA8PSBuciA8IGhlaWdodCBhbmQgMCA8PSBuYyA8IHdpZHRoOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgaWYgY29tcF9pZFtucl1bbmNdICE9IG91dGVyX2lkIGFuZCBjb21wX2lkW25yXVtuY10gIT0gaW5uZXJfaWQ6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaXNfY2hpbGQgPSBGYWxzZQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IGlzX2NoaWxkOgogICAgICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICAgICAgaWYgaXNfY2hpbGQ6CiAgICAgICAgICAgICAgICAgICAgbm9kZXNbb3V0ZXJfaWRdWyJjaGlsZHJlbiJdLmFwcGVuZChpbm5lcl9pZCkKICAgIAogICAgIyBDb21wdXRlIGFkamFjZW5jeQogICAgYWRqX3NldCA9IHNldCgpCiAgICBmb3IgciBpbiByYW5nZShoZWlnaHQpOgogICAgICAgIGZvciBjIGluIHJhbmdlKHdpZHRoKToKICAgICAgICAgICAgY2lkID0gY29tcF9pZFtyXVtjXQogICAgICAgICAgICBmb3IgZHIsIGRjIGluIF9PUlRIOgogICAgICAgICAgICAgICAgbnIsIG5jID0gciArIGRyLCBjICsgZGMKICAgICAgICAgICAgICAgIGlmIDAgPD0gbnIgPCBoZWlnaHQgYW5kIDAgPD0gbmMgPCB3aWR0aDoKICAgICAgICAgICAgICAgICAgICBuaWQgPSBjb21wX2lkW25yXVtuY10KICAgICAgICAgICAgICAgICAgICBpZiBjaWQgIT0gbmlkOgogICAgICAgICAgICAgICAgICAgICAgICBwYWlyID0gKG1pbihjaWQsIG5pZCksIG1heChjaWQsIG5pZCkpCiAgICAgICAgICAgICAgICAgICAgICAgIGFkal9zZXQuYWRkKHBhaXIpCiAgICAKICAgIHJldHVybiB7CiAgICAgICAgIm5vZGVzIjogbm9kZXMsCiAgICAgICAgImFkamFjZW5jeV9saXN0Ijogc29ydGVkKGFkal9zZXQpLAogICAgfQoKCmRlZiBjb21wYWN0X3NlZ21lbnRhdGlvbl9zdW1tYXJ5KHNlZzogZGljdCwgbWF4X25vZGVzOiBpbnQgPSAyMCkgLT4gc3RyOgogICAgIiIiQ3JlYXRlIGEgY29tcGFjdCB0ZXh0IHN1bW1hcnkgb2Ygc2VnbWVudGF0aW9uIGZvciBMTE0gY29udGV4dC4iIiIKICAgIG5vZGVzID0gc2VnLmdldCgibm9kZXMiLCBbXSkKICAgIGFkaiA9IHNlZy5nZXQoImFkamFjZW5jeV9saXN0IiwgW10pCiAgICAKICAgIGxpbmVzID0gW2YiT2JqZWN0czoge2xlbihub2Rlcyl9LCBBZGphY2VuY2llczoge2xlbihhZGopfSJdCiAgICBmb3Igbm9kZSBpbiBub2Rlc1s6bWF4X25vZGVzXToKICAgICAgICBib3VuZGFyeV9zdHIgPSAiLCIuam9pbihmIntyfSx7Y30iIGZvciByLCBjIGluIG5vZGVbImJvdW5kYXJ5Il1bOjRdKQogICAgICAgIGNoaWxkcmVuX3N0ciA9IGYiLCBjaGlsZHJlbj1beycsJy5qb2luKG1hcChzdHIsIG5vZGVbJ2NoaWxkcmVuJ10pKX1dIiBpZiBub2RlWyJjaGlsZHJlbiJdIGVsc2UgIiIKICAgICAgICBsaW5lcy5hcHBlbmQoCiAgICAgICAgICAgIGYiICBvYmoje25vZGVbJ2lkJ119OiBjb2xvcj17bm9kZVsnY29sb3InXX0sIHNpemU9e25vZGVbJ3BpeGVscyddfXB4LCAiCiAgICAgICAgICAgIGYiaGFzaD17bm9kZVsnaGFzaCddWzo4XX0sIGNvcm5lcnM9W3tib3VuZGFyeV9zdHJ9XXtjaGlsZHJlbl9zdHJ9IgogICAgICAgICkKICAgIGlmIGxlbihub2RlcykgPiBtYXhfbm9kZXM6CiAgICAgICAgbGluZXMuYXBwZW5kKGYiICAuLi4gYW5kIHtsZW4obm9kZXMpIC0gbWF4X25vZGVzfSBtb3JlIG9iamVjdHMiKQogICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykK"))
(agent_dir / "llm_policy.py").write_bytes(base64.b64decode("..."))
(agent_dir / "llm_agent.py").write_bytes(base64.b64decode("IiIiQVJDLUFHSS0zIEFnZW50IGFkYXB0ZXIgZm9yIHRoZSBMTE0tYmFzZWQgcG9saWN5LgoKUGx1Z3MgTExNUG9saWN5IGludG8gdGhlIEFSQy1BR0ktMy1BZ2VudHMgcmVmZXJlbmNlIGZyYW1ld29yay4KVGhlIGFnZW50IGNhbGxzIHRoZSBMTE0gdG8gZ2V0IGFjdGlvbnMsIHdpdGggYW50aS1sb29wIGRldGVjdGlvbgphbmQgYWN0aW9uIGJhdGNoaW5nIGZvciBlZmZpY2llbmN5LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSB0aHJlYWRpbmcgaW1wb3J0IFJMb2NrCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCiMgRW5zdXJlIGltcG9ydHMgd29yayBib3RoIGluIGNvbXBldGl0aW9uIGFuZCBsb2NhbGx5Cl9kaXIgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkKX3BhcmVudCA9IG9zLnBhdGguZGlybmFtZShfZGlyKQpmb3IgcCBpbiAoX2RpciwgX3BhcmVudCk6CiAgICBpZiBwIG5vdCBpbiBzeXMucGF0aDoKICAgICAgICBzeXMucGF0aC5pbnNlcnQoMCwgcCkKCmZyb20gYXJjZW5naW5lIGltcG9ydCBBY3Rpb25JbnB1dCwgRnJhbWVEYXRhLCBHYW1lQWN0aW9uLCBHYW1lU3RhdGUKZnJvbSBhZ2VudHMuYWdlbnQgaW1wb3J0IEFnZW50Cgp0cnk6CiAgICBmcm9tIC5sbG1fcG9saWN5IGltcG9ydCBMTE1Qb2xpY3ksIExMTUNvbmZpZywgRnJhbWVJbmZvCiAgICBmcm9tIC5ncmlkX3V0aWxzIGltcG9ydCBmb3JtYXRfZ3JpZF9hc2NpaQpleGNlcHQgSW1wb3J0RXJyb3I6CiAgICBmcm9tIGxsbV9wb2xpY3kgaW1wb3J0IExMTVBvbGljeSwgTExNQ29uZmlnLCBGcmFtZUluZm8KICAgIGZyb20gZ3JpZF91dGlscyBpbXBvcnQgZm9ybWF0X2dyaWRfYXNjaWkKCgpkZWYgX2V4dHJhY3RfZ3JpZChmcmFtZV9kYXRhOiBGcmFtZURhdGEpIC0+IHR1cGxlW3R1cGxlW2ludCwgLi4uXSwgLi4uXToKICAgICIiIkV4dHJhY3QgdGhlIG51bWVyaWMgZ3JpZCBmcm9tIGEgRnJhbWVEYXRhIG9iamVjdC4iIiIKICAgICMgVHJ5IG11bHRpcGxlIHBhdGhzIHRvIGdldCB0aGUgZ3JpZCBkYXRhCiAgICBmb3IgYXR0cl9wYXRoIGluIFsKICAgICAgICBsYW1iZGEgZjogZi5yYXdfc3RhdGUuZGF0YSwKICAgICAgICBsYW1iZGEgZjogZi5yYXdfc3RhdGUucGxhbmVzWzBdIGlmIGYucmF3X3N0YXRlLnBsYW5lcyBlbHNlIE5vbmUsCiAgICAgICAgbGFtYmRhIGY6IGYucGxhbmVzWzBdIGlmIGhhc2F0dHIoZiwgJ3BsYW5lcycpIGFuZCBmLnBsYW5lcyBlbHNlIE5vbmUsCiAgICBdOgogICAgICAgIHRyeToKICAgICAgICAgICAgZGF0YSA9IGF0dHJfcGF0aChmcmFtZV9kYXRhKQogICAgICAgICAgICBpZiBkYXRhIGlzIE5vbmU6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBpZiBoYXNhdHRyKGRhdGEsICd0b2xpc3QnKToKICAgICAgICAgICAgICAgIHJvd3MgPSBkYXRhLnRvbGlzdCgpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICByb3dzID0gbGlzdChkYXRhKQogICAgICAgICAgICBpZiByb3dzIGFuZCByb3dzWzBdOgogICAgICAgICAgICAgICAgcmV0dXJuIHR1cGxlKHR1cGxlKGludChjKSBmb3IgYyBpbiByb3cpIGZvciByb3cgaW4gcm93cykKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBjb250aW51ZQogICAgIyBBYnNvbHV0ZSBmYWxsYmFjawogICAgcmV0dXJuICgoMCwpICogNjQsKSAqIDY0CgoKZGVmIF9mcmFtZV90b19pbmZvKGZyYW1lX2RhdGE6IEZyYW1lRGF0YSkgLT4gRnJhbWVJbmZvOgogICAgIiIiQ29udmVydCBGcmFtZURhdGEgdG8gb3VyIEZyYW1lSW5mby4iIiIKICAgIGdyaWQgPSBfZXh0cmFjdF9ncmlkKGZyYW1lX2RhdGEpCiAgICAKICAgICMgU3RhdGUgbmFtZQogICAgdHJ5OgogICAgICAgIHN0YXRlID0gZnJhbWVfZGF0YS5zdGF0ZQogICAgICAgIHN0YXRlX25hbWUgPSBzdGF0ZS5uYW1lIGlmIGhhc2F0dHIoc3RhdGUsICduYW1lJykgZWxzZSBzdHIoc3RhdGUpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHN0YXRlX25hbWUgPSAiUExBWUlORyIKICAgIAogICAgIyBMZXZlbHMgY29tcGxldGVkCiAgICB0cnk6CiAgICAgICAgbGV2ZWxzX2NvbXBsZXRlZCA9IGludChmcmFtZV9kYXRhLmxldmVsc19jb21wbGV0ZWQpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGxldmVsc19jb21wbGV0ZWQgPSAwCiAgICAKICAgICMgQXZhaWxhYmxlIGFjdGlvbnMKICAgIGF2YWlsYWJsZSA9IFtdCiAgICB0cnk6CiAgICAgICAgZm9yIGFjdF9pZCBpbiBmcmFtZV9kYXRhLmF2YWlsYWJsZV9hY3Rpb25zOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBhY3QgPSBHYW1lQWN0aW9uLmZyb21faWQoaW50KGFjdF9pZCkpCiAgICAgICAgICAgICAgICBpZiBhY3QubmFtZSAhPSAiUkVTRVQiOgogICAgICAgICAgICAgICAgICAgIGF2YWlsYWJsZS5hcHBlbmQoYWN0Lm5hbWUpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICBwYXNzCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGF2YWlsYWJsZSA9IFsiQUNUSU9OMSIsICJBQ1RJT04yIiwgIkFDVElPTjMiLCAiQUNUSU9ONCIsICJBQ1RJT041IiwgIkFDVElPTjYiLCAiQUNUSU9ONyJdCiAgICAKICAgICMgU3RlcCBudW1iZXIKICAgIHRyeToKICAgICAgICBzdGVwID0gaW50KGZyYW1lX2RhdGEuc3RlcCkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgc3RlcCA9IDAKICAgIAogICAgcmV0dXJuIEZyYW1lSW5mbygKICAgICAgICBncmlkPWdyaWQsCiAgICAgICAgc3RlcD1zdGVwLAogICAgICAgIGxldmVsPWxldmVsc19jb21wbGV0ZWQgKyAxLAogICAgICAgIHN0YXRlX25hbWU9c3RhdGVfbmFtZSwKICAgICAgICBsZXZlbHNfY29tcGxldGVkPWxldmVsc19jb21wbGV0ZWQsCiAgICAgICAgYXZhaWxhYmxlX2FjdGlvbnM9YXZhaWxhYmxlLAogICAgKQoKCmNsYXNzIExMTUFnZW50KEFnZW50KToKICAgICIiIkxMTS1wb3dlcmVkIGFnZW50IGZvciBBUkMtQUdJLTMuIiIiCiAgICAKICAgIF9BQ1RJT05fTE9DSyA9IFJMb2NrKCkKICAgIAogICAgQUNUSU9OX0JVREdFVCA9IDQwMAogICAgTUFYX0FDVElPTlMgPSBBQ1RJT05fQlVER0VUCiAgICAKICAgICMgU29mdCB0aW1lIGxpbWl0IHBlciBnYW1lIChzZWNvbmRzKQogICAgIyAxMTAgZ2FtZXMgw5cgNCBtaW4gPSA0NDAgbWluIOKJiCA3LjMgaG91cnMg4oCUIGZpdHMgaW4gOWggd2l0aCBtYXJnaW4KICAgIEdBTUVfVElNRV9MSU1JVCA9IDI0MAogICAgCiAgICBkZWYgX19pbml0X18oc2VsZiwgKmFyZ3M6IEFueSwgKiprd2FyZ3M6IEFueSkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCphcmdzLCAqKmt3YXJncykKICAgICAgICBzZWxmLmNvbmZpZyA9IExMTUNvbmZpZy5mcm9tX2VudigpCiAgICAgICAgc2VsZi5wb2xpY3kgPSBMTE1Qb2xpY3koc2VsZi5jb25maWcpCiAgICAgICAgc2VsZi5faG9sZHNfYWN0aW9uX2xvY2sgPSBGYWxzZQogICAgICAgIHNlbGYuX2dhbWVfc3RhcnRfdGltZSA9IHRpbWUudGltZSgpCiAgICAgICAgc2VsZi5fYWN0aW9uc19pbl9nYW1lID0gMAogICAgICAgIHNlbGYuX2ZpcnN0X2FjdGlvbiA9IFRydWUKICAgIAogICAgQHByb3BlcnR5CiAgICBkZWYgbmFtZShzZWxmKSAtPiBzdHI6CiAgICAgICAgcmV0dXJuIGYie3N1cGVyKCkubmFtZX0ubGxtLXYyIgogICAgCiAgICBkZWYgaXNfZG9uZShzZWxmLCBmcmFtZXM6IGxpc3RbRnJhbWVEYXRhXSwgbGF0ZXN0X2ZyYW1lOiBGcmFtZURhdGEpIC0+IGJvb2w6CiAgICAgICAgIiIiU3RvcCBjb25kaXRpb25zOiB3aW4sIGJ1ZGdldCwgb3IgdGltZSBsaW1pdC4iIiIKICAgICAgICBpZiBzZWxmLl9hY3Rpb25zX2luX2dhbWUgPj0gc2VsZi5BQ1RJT05fQlVER0VUOgogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIHRyeToKICAgICAgICAgICAgaWYgbGF0ZXN0X2ZyYW1lLnN0YXRlIGlzIEdhbWVTdGF0ZS5XSU46CiAgICAgICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKICAgICAgICBpZiB0aW1lLnRpbWUoKSAtIHNlbGYuX2dhbWVfc3RhcnRfdGltZSA+IHNlbGYuR0FNRV9USU1FX0xJTUlUOgogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIHJldHVybiBGYWxzZQogICAgCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgX3RvX2dhbWVfYWN0aW9uKHByb3Bvc2FsOiBkaWN0KSAtPiBHYW1lQWN0aW9uOgogICAgICAgICIiIkNvbnZlcnQgcG9saWN5IG91dHB1dCB0byBHYW1lQWN0aW9uIGVudW0uIiIiCiAgICAgICAgYWN0aW9uX25hbWUgPSBwcm9wb3NhbC5nZXQoIm5hbWUiLCAiUkVTRVQiKQogICAgICAgIAogICAgICAgIHRyeToKICAgICAgICAgICAgYWN0aW9uID0gR2FtZUFjdGlvblthY3Rpb25fbmFtZV0KICAgICAgICBleGNlcHQgS2V5RXJyb3I6CiAgICAgICAgICAgIGFjdGlvbiA9IEdhbWVBY3Rpb24uUkVTRVQKICAgICAgICAKICAgICAgICBpZiBhY3Rpb24uaXNfY29tcGxleCgpOgogICAgICAgICAgICB4ID0gcHJvcG9zYWwuZ2V0KCJ4IiwgMCkgb3IgMAogICAgICAgICAgICB5ID0gcHJvcG9zYWwuZ2V0KCJ5IiwgMCkgb3IgMAogICAgICAgICAgICBhY3Rpb24uc2V0X2RhdGEoeyJ4IjogbWF4KDAsIG1pbig2MywgaW50KHgpKSksICJ5IjogbWF4KDAsIG1pbig2MywgaW50KHkpKSl9KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGFjdGlvbi5zZXRfZGF0YSh7fSkKICAgICAgICAKICAgICAgICBhY3Rpb24ucmVhc29uaW5nID0gcHJvcG9zYWwuZ2V0KCJyZWFzb25pbmciLCB7fSkKICAgICAgICByZXR1cm4gYWN0aW9uCiAgICAKICAgIGRlZiBjaG9vc2VfYWN0aW9uKAogICAgICAgIHNlbGYsIGZyYW1lczogbGlzdFtGcmFtZURhdGFdLCBsYXRlc3RfZnJhbWU6IEZyYW1lRGF0YQogICAgKSAtPiBHYW1lQWN0aW9uOgogICAgICAgICIiIk1haW4gZW50cnkgcG9pbnQgY2FsbGVkIGJ5IHRoZSBmcmFtZXdvcmsgZWFjaCB0dXJuLiIiIgogICAgICAgIHNlbGYuX0FDVElPTl9MT0NLLmFjcXVpcmUoKQogICAgICAgIHNlbGYuX2hvbGRzX2FjdGlvbl9sb2NrID0gVHJ1ZQogICAgICAgIAogICAgICAgIHRyeToKICAgICAgICAgICAgZnJhbWVfaW5mbyA9IF9mcmFtZV90b19pbmZvKGxhdGVzdF9mcmFtZSkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgSW5pdGlhbGl6ZSBvbiBmaXJzdCBjYWxsCiAgICAgICAgICAgIGlmIHNlbGYuX2ZpcnN0X2FjdGlvbjoKICAgICAgICAgICAgICAgIHNlbGYuX2ZpcnN0X2FjdGlvbiA9IEZhbHNlCiAgICAgICAgICAgICAgICBzZWxmLl9nYW1lX3N0YXJ0X3RpbWUgPSB0aW1lLnRpbWUoKQogICAgICAgICAgICAgICAgc2VsZi5wb2xpY3kucmVzZXRfZ2FtZSgpCiAgICAgICAgICAgIAogICAgICAgICAgICAjIEF1dG8tcmVzZXQgb24gR0FNRV9PVkVSCiAgICAgICAgICAgIGlmIGZyYW1lX2luZm8uc3RhdGVfbmFtZSA9PSAiR0FNRV9PVkVSIjoKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBzZWxmLnBvbGljeS5ub3RlX2dhbWVfb3ZlcihmcmFtZV9pbmZvKQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICBwYXNzCiAgICAgICAgICAgICAgICBhY3Rpb24gPSBHYW1lQWN0aW9uLlJFU0VUCiAgICAgICAgICAgICAgICBhY3Rpb24uc2V0X2RhdGEoe30pCiAgICAgICAgICAgICAgICBhY3Rpb24ucmVhc29uaW5nID0geyJhdXRvX3Jlc2V0IjogVHJ1ZSwgImdhbWVfb3ZlciI6IFRydWV9CiAgICAgICAgICAgICAgICByZXR1cm4gYWN0aW9uCiAgICAgICAgICAgIAogICAgICAgICAgICAjIENhbGwgdGhlIExMTSBwb2xpY3kKICAgICAgICAgICAgcHJvcG9zYWwgPSBzZWxmLnBvbGljeS5jaG9vc2VfYWN0aW9uKGZyYW1lX2luZm8pCiAgICAgICAgICAgIHNlbGYuX2FjdGlvbnNfaW5fZ2FtZSArPSAxCiAgICAgICAgICAgIAogICAgICAgICAgICByZXR1cm4gc2VsZi5fdG9fZ2FtZV9hY3Rpb24ocHJvcG9zYWwpCiAgICAgICAgICAgIAogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgaW1wb3J0IHRyYWNlYmFjawogICAgICAgICAgICBwcmludChmIltMTE1BZ2VudF0gRXJyb3I6IHtlfSIpCiAgICAgICAgICAgIHRyYWNlYmFjay5wcmludF9leGMoKQogICAgICAgICAgICBhY3Rpb24gPSBHYW1lQWN0aW9uLlJFU0VUCiAgICAgICAgICAgIGFjdGlvbi5zZXRfZGF0YSh7fSkKICAgICAgICAgICAgYWN0aW9uLnJlYXNvbmluZyA9IHsiZXJyb3IiOiBzdHIoZSl9CiAgICAgICAgICAgIHJldHVybiBhY3Rpb24KICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICBpZiBzZWxmLl9ob2xkc19hY3Rpb25fbG9jazoKICAgICAgICAgICAgICAgIHNlbGYuX2hvbGRzX2FjdGlvbl9sb2NrID0gRmFsc2UKICAgICAgICAgICAgICAgIHNlbGYuX0FDVElPTl9MT0NLLnJlbGVhc2UoKQogICAgCiAgICBkZWYgdGFrZV9hY3Rpb24oc2VsZiwgYWN0aW9uOiBHYW1lQWN0aW9uKTogICMgdHlwZTogaWdub3JlW292ZXJyaWRlXQogICAgICAgICIiIkV4ZWN1dGUgYWN0aW9uIGFuZCByZWNvcmQgdGhlIHJlc3VsdC4iIiIKICAgICAgICB0cnk6CiAgICAgICAgICAgIGZyYW1lID0gc3VwZXIoKS50YWtlX2FjdGlvbihhY3Rpb24pCiAgICAgICAgICAgIGlmIGZyYW1lIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgZnJhbWUuYWN0aW9uX2lucHV0ID0gQWN0aW9uSW5wdXQoCiAgICAgICAgICAgICAgICAgICAgaWQ9YWN0aW9uLAogICAgICAgICAgICAgICAgICAgIGRhdGE9YWN0aW9uLmFjdGlvbl9kYXRhLm1vZGVsX2R1bXAoKSwKICAgICAgICAgICAgICAgICAgICByZWFzb25pbmc9Z2V0YXR0cihhY3Rpb24sICJyZWFzb25pbmciLCBOb25lKSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgcmV0dXJuIGZyYW1lCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgaWYgZ2V0YXR0cihzZWxmLCAiX2hvbGRzX2FjdGlvbl9sb2NrIiwgRmFsc2UpOgogICAgICAgICAgICAgICAgc2VsZi5faG9sZHNfYWN0aW9uX2xvY2sgPSBGYWxzZQogICAgICAgICAgICAgICAgc2VsZi5fQUNUSU9OX0xPQ0sucmVsZWFzZSgpCiAgICAKICAgIGRlZiBmaW5hbGl6ZV9kaWFnbm9zdGljcyhzZWxmKSAtPiBOb25lOgogICAgICAgICIiIkxvZyBmaW5hbCBzdGF0cyBmb3IgdGhpcyBnYW1lLiIiIgogICAgICAgIGVsYXBzZWQgPSB0aW1lLnRpbWUoKSAtIHNlbGYuX2dhbWVfc3RhcnRfdGltZQogICAgICAgIHByaW50KGYiW0xMTUFnZW50XSBHYW1lIGRvbmU6ICIKICAgICAgICAgICAgICBmIntzZWxmLl9hY3Rpb25zX2luX2dhbWV9IGFjdGlvbnMgaW4ge2VsYXBzZWQ6LjBmfXMsICIKICAgICAgICAgICAgICBmIntzZWxmLnBvbGljeS50b3RhbF90b2tlbnN9IHRva2VucyAoe3NlbGYucG9saWN5LnRvdGFsX2NhbGxzfSBjYWxscywgIgogICAgICAgICAgICAgIGYie3NlbGYucG9saWN5LmNsaWVudC50b3RhbF90aW1lOi4wZn1zIExMTSB0aW1lKSwgIgogICAgICAgICAgICAgIGYie2xlbihzZWxmLnBvbGljeS50cmFja2VyLnVuaXF1ZV9zdGF0ZXNfc2Vlbil9IHVuaXF1ZSBzdGF0ZXMsICIKICAgICAgICAgICAgICBmIntzZWxmLnBvbGljeS50cmFja2VyLmdhbWVfb3Zlcl9jb3VudH0gZGVhdGhzIikK"))
(agent_dir / "value_gate.py").write_bytes(base64.b64decode("IiIiVmFsdWUgZ2F0ZSBzdGFudSBncnkgKHY2LjUpIOKAlCBwb215c8WCIHogaWNoIHY1LjEsIGJleiBpY2ggd2FkLgoKUsOzxbxuaWNlIHdvYmVjIGBsbG1fcG9saWN5X3Y1XzEucHlgICgxNjc1YWE0KToKLSBCcmFrIHdhZyA9PiBnYXRlIGplc3QgV1nFgcSEQ1pPTlkgKHUgbmljaDogbG9zb3dhIGluaWNqYWxpemFjamEgPSBzenVtKS4KLSBQcmVkeWtjamEgd3ltYWdhIEpTT04geiB3YWdhbWkgKHogdHJhaW5fdmFsdWVfbW9kZWwucHkpIHBvZGFueW0gcHJ6ZXoKICBlbnYgVkFMVUVfTU9ERUxfSlNPTjsga2HFvGRhIHVzdGVya2EgPSBOb25lIChjaXN6YSksIG5pZ2R5IGxvc293eSB3eW5pay4KLSBTcGxpdCB0cmVuaW5nb3d5IHBlciBHUkEgamVzdCB3IHRyYWluZXJ6ZSDigJQgbmllIG1hIHd5Y2lla3Ugd2llcnN6b3dlZ28sCiAga3TDs3J5IHphd3nFvGHFgiDigJ5hY2MgMC45MTMiIHcgUG9rZW1vbmllIChzdGFueSBrb2xlam5lIHPEhSBuaWVtYWwgaWRlbnR5Y3puZSkuCgpDZWNoeTogMTIgc3lnbmHFgsOzdyB6IEZyYW1lSW5mbyBpIHN0YW51IHBvbGl0eWtpLCB3c3p5c3RraWUgdyBbMCwgMV0uCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQpmcm9tIHR5cGluZyBpbXBvcnQgVFlQRV9DSEVDS0lORywgQW55CgppZiBUWVBFX0NIRUNLSU5HOiAgIyB0eWxrbyBkbGEgdHlww7N3OyB1bmlrYSBjeWtsaWN6bmVnbyBpbXBvcnR1CiAgICBmcm9tIC4gaW1wb3J0IGxsbV9wb2xpY3kgYXMgX2xwCgpGRUFUVVJFUzogbGlzdFtzdHJdID0gWwogICAgInJvd3MiLCAgICAgICAgICAgICAgIyBncmlkLnJvd3MgLyA2NAogICAgImNvbHMiLCAgICAgICAgICAgICAgIyBncmlkLmNvbHMgLyA2NAogICAgInNpemUiLCAgICAgICAgICAgICAgIyByb3dzKmNvbHMgLyA0MDk2CiAgICAidW5pcXVlX2NvbG9ycyIsICAgICAjIHVuaWthbG5lIGtvbG9yeSAvIDE2CiAgICAiZGVuc2l0eSIsICAgICAgICAgICAjIHVkemlhxYIgcMOzbCAhPSAwCiAgICAib2JqZWN0cyIsICAgICAgICAgICAjIG9iaWVrdHkgeiBzZWdtZW50YWNqaSAvIDMwCiAgICAibGV2ZWxzX2NvbXBsZXRlZCIsICAjIGxldmVsc19jb21wbGV0ZWQgLyA1CiAgICAiYWN0aW9uc19vbl9sZXZlbCIsICAjIGFrY2plIG5hIHBvemlvbWllIC8gMTIwCiAgICAibm9vcF9yYXRpbyIsICAgICAgICAjIG5vLW9weSB3IG9rbmllIDEwCiAgICAidG9rZW5zX2xldmVsIiwgICAgICAjIHRva2VueSBuYSBwb3ppb21pZSAvIDEwMGsKICAgICJhdWRpdHNfb25fbGV2ZWwiLCAgICMgYXVkeXR5IG5hIHBvemlvbWllIC8gMwogICAgInN0ZXBzIiwgICAgICAgICAgICAgIyBmcmFtZS5zdGVwIC8gMTE1IChsaW1pdCBrcm9rw7N3IHBvemlvbXUpCl0KCl9PQkpfUkUgPSByZS5jb21waWxlKHIiT2JqZWN0czpccyooXGQrKSIpCgoKZGVmIGV4dHJhY3RfZmVhdHVyZXMocG9saWN5OiBBbnksIGZyYW1lOiBBbnkpIC0+IGxpc3RbZmxvYXRdOgogICAgIiIiMTIgY2VjaCBzdGFudSB3IHN0YcWCZWoga29sZWpub8WbY2kgRkVBVFVSRVMuIERldGVybWluaXN0eWN6bmUuIiIiCiAgICByb3dzLCBjb2xzID0gZnJhbWUuc2hhcGUKICAgIGdyaWQgPSBmcmFtZS5ncmlkCiAgICBjb2xvcnM6IHNldFtpbnRdID0gc2V0KCkKICAgIG5vbnplcm8gPSAwCiAgICB0b3RhbCA9IDAKICAgIGZvciByb3cgaW4gZ3JpZDoKICAgICAgICBmb3IgY2VsbCBpbiByb3c6CiAgICAgICAgICAgIHRvdGFsICs9IDEKICAgICAgICAgICAgaWYgY2VsbDoKICAgICAgICAgICAgICAgIG5vbnplcm8gKz0gMQogICAgICAgICAgICAgICAgY29sb3JzLmFkZChjZWxsKQoKICAgIHNlZyA9IGZyYW1lLnNlZ21lbnRhdGlvbl9zdW1tYXJ5KCkKICAgIG1hdGNoID0gX09CSl9SRS5zZWFyY2goc2VnKQogICAgb2JqZWN0cyA9IGludChtYXRjaC5ncm91cCgxKSkgaWYgbWF0Y2ggZWxzZSAwCgogICAgbm9vcF90b3RhbCA9IGxlbihnZXRhdHRyKHBvbGljeSwgIl9ub29wX3JlY2VudCIsIFtdKSBvciBbXSkKICAgIG5vb3BzID0gc3VtKDEgZm9yIF8sIGNoYW5nZWQgaW4gZ2V0YXR0cihwb2xpY3ksICJfbm9vcF9yZWNlbnQiLCBbXSkgaWYgbm90IGNoYW5nZWQpCiAgICBub29wX3JhdGlvID0gbm9vcHMgLyBub29wX3RvdGFsIGlmIG5vb3BfdG90YWwgZWxzZSAwLjAKCiAgICBsZXZlbF90b2tlbnMgPSBtYXgoMCwgcG9saWN5LnRvdGFsX3Rva2VucyAtIHBvbGljeS5fbGV2ZWxfc3RhcnRfdG9rZW5zKSBcCiAgICAgICAgaWYgaGFzYXR0cihwb2xpY3ksICJfbGV2ZWxfc3RhcnRfdG9rZW5zIikgZWxzZSAwCgogICAgcmV0dXJuIFsKICAgICAgICBtaW4oMS4wLCByb3dzIC8gNjQuMCksCiAgICAgICAgbWluKDEuMCwgY29scyAvIDY0LjApLAogICAgICAgIG1pbigxLjAsIChyb3dzICogY29scykgLyA0MDk2LjApLAogICAgICAgIG1pbigxLjAsIGxlbihjb2xvcnMpIC8gMTYuMCksCiAgICAgICAgbm9uemVybyAvIHRvdGFsIGlmIHRvdGFsIGVsc2UgMC4wLAogICAgICAgIG1pbigxLjAsIG9iamVjdHMgLyAzMC4wKSwKICAgICAgICBtaW4oMS4wLCBmcmFtZS5sZXZlbHNfY29tcGxldGVkIC8gNS4wKSwKICAgICAgICBtaW4oMS4wLCBnZXRhdHRyKHBvbGljeSwgIl9hY3Rpb25zX29uX2xldmVsIiwgMCkgLyAxMjAuMCksCiAgICAgICAgbm9vcF9yYXRpbywKICAgICAgICBtaW4oMS4wLCBsZXZlbF90b2tlbnMgLyAxMDBfMDAwLjApLAogICAgICAgIG1pbigxLjAsIGdldGF0dHIocG9saWN5LCAiX2F1ZGl0c19vbl9sZXZlbCIsIDApIC8gMy4wKSwKICAgICAgICBtaW4oMS4wLCBmcmFtZS5zdGVwIC8gMTE1LjApLAogICAgXQoKCmNsYXNzIFZhbHVlR2F0ZToKICAgICIiIk9wY2pvbmFsbnksIHd5dHJlbm93YW55IG9jZW5pYWrEhWN5IHN0YW51LiBgb2s9RmFsc2VgID0+IGNhxYJraWVtIHd5xYLEhWN6b255LiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB3ZWlnaHRzOiBkaWN0W3N0ciwgbGlzdFtsaXN0W2Zsb2F0XV1dLCB0aHJlc2hvbGRzOiBkaWN0W3N0ciwgZmxvYXRdKToKICAgICAgICBzZWxmLlcwID0gd2VpZ2h0c1siVzAiXQogICAgICAgIHNlbGYuYjAgPSB3ZWlnaHRzWyJiMCJdCiAgICAgICAgc2VsZi5XMSA9IHdlaWdodHNbIlcxIl0KICAgICAgICBzZWxmLmIxID0gd2VpZ2h0c1siYjEiXQogICAgICAgIHNlbGYuVzIgPSB3ZWlnaHRzWyJXMiJdCiAgICAgICAgc2VsZi5iMiA9IHdlaWdodHNbImIyIl0KICAgICAgICBzZWxmLmZlYXR1cmVzOiBsaXN0W3N0cl0gPSBsaXN0KHdlaWdodHMuZ2V0KCJmZWF0dXJlcyIsIEZFQVRVUkVTKSkKICAgICAgICBzZWxmLmxvdyA9IGZsb2F0KHRocmVzaG9sZHMuZ2V0KCJsb3ciLCAwLjMpKQogICAgICAgIHNlbGYuaGlnaCA9IGZsb2F0KHRocmVzaG9sZHMuZ2V0KCJoaWdoIiwgMC43KSkKICAgICAgICBzZWxmLm9rID0gVHJ1ZQoKICAgIEBjbGFzc21ldGhvZAogICAgZGVmIGxvYWQoY2xzLCBwYXRoOiBzdHIgfCBOb25lKSAtPiAiVmFsdWVHYXRlIHwgTm9uZSI6CiAgICAgICAgIiIiV2N6eXRhaiB3YWdpIHogSlNPTi4gQnJhayDFm2NpZcW8a2kvcGxpa3UvYsWCxJlkdSA9PiBOb25lIChnYXRlIE9GRikuIiIiCiAgICAgICAgaWYgbm90IHBhdGg6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgdHJ5OgogICAgICAgICAgICB3aXRoIG9wZW4ocGF0aCwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZmg6CiAgICAgICAgICAgICAgICBibG9iID0ganNvbi5sb2FkKGZoKQogICAgICAgICAgICByZXR1cm4gY2xzKGJsb2JbIndlaWdodHMiXSwgYmxvYi5nZXQoInRocmVzaG9sZHMiLCB7fSkpCiAgICAgICAgZXhjZXB0IChPU0Vycm9yLCBLZXlFcnJvciwgVmFsdWVFcnJvciwgVHlwZUVycm9yKToKICAgICAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBAY2xhc3NtZXRob2QKICAgIGRlZiBmcm9tX2VudihjbHMpIC0+ICJWYWx1ZUdhdGUgfCBOb25lIjoKICAgICAgICByZXR1cm4gY2xzLmxvYWQob3MuZW52aXJvbi5nZXQoIlZBTFVFX01PREVMX0pTT04iLCAiIikpCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIF9mb3J3YXJkX3goeDogbGlzdFtmbG9hdF0sIFcwOiBsaXN0W2xpc3RbZmxvYXRdXSwgYjA6IGxpc3RbZmxvYXRdLAogICAgICAgICAgICAgICAgICAgVzE6IGxpc3RbbGlzdFtmbG9hdF1dLCBiMTogbGlzdFtmbG9hdF0sCiAgICAgICAgICAgICAgICAgICBXMjogbGlzdFtsaXN0W2Zsb2F0XV0sIGIyOiBsaXN0W2Zsb2F0XSkgLT4gZmxvYXQ6CiAgICAgICAgaDEgPSBbbWF4KDAuMCwgc3VtKHdbal0gKiB4W2pdIGZvciBqIGluIHJhbmdlKGxlbih4KSkpICsgYikKICAgICAgICAgICAgICBmb3IgdywgYiBpbiB6aXAoVzAsIGIwKV0KICAgICAgICBoMiA9IFttYXgoMC4wLCBzdW0od1tqXSAqIGgxW2pdIGZvciBqIGluIHJhbmdlKGxlbihoMSkpKSArIGIpCiAgICAgICAgICAgICAgZm9yIHcsIGIgaW4gemlwKFcxLCBiMSldCiAgICAgICAgeiA9IHN1bSh3WzBdICogaDJbal0gZm9yIGosIHcgaW4gZW51bWVyYXRlKFcyKSkgKyBiMlswXQogICAgICAgIHJldHVybiAxLjAgLyAoMS4wICsgcG93KDIuNzE4MjgxODI4NDU5MDQ1LCAtbWF4KC0yMC4wLCBtaW4oMjAuMCwgeikpKSkKCiAgICBkZWYgX2ZvcndhcmQoc2VsZiwgeDogbGlzdFtmbG9hdF0pIC0+IGZsb2F0OgogICAgICAgICMgQ3p5c3R5IFB5dGhvbjogc2llxIcgbWEgMTI4KzY0IG5ldXJvbsOzdywgd3l3b8WCeXdhbmEgbWF4IHJheiBuYSB0dXLEmSBMTE0uCiAgICAgICAgcmV0dXJuIHNlbGYuX2ZvcndhcmRfeCh4LCBzZWxmLlcwLCBzZWxmLmIwLCBzZWxmLlcxLCBzZWxmLmIxLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5XMiwgc2VsZi5iMikKCiAgICBkZWYgdmFsdWUoc2VsZiwgcG9saWN5OiBBbnksIGZyYW1lOiBBbnkpIC0+IGZsb2F0IHwgTm9uZToKICAgICAgICAiIiJXYXJ0b8WbxIcgc3RhbnUgdyBbMCwxXSBhbGJvIE5vbmUsIGdkeSBnYXRlIHd5xYLEhWN6b255L3VzemtvZHpvbnkuIiIiCiAgICAgICAgaWYgbm90IGdldGF0dHIoc2VsZiwgIm9rIiwgRmFsc2UpOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIHRyeToKICAgICAgICAgICAgeCA9IGV4dHJhY3RfZmVhdHVyZXMocG9saWN5LCBmcmFtZSkKICAgICAgICAgICAgaWYgbGVuKHgpICE9IGxlbihzZWxmLmZlYXR1cmVzKToKICAgICAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgICAgIHJldHVybiBzZWxmLl9mb3J3YXJkKHgpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgc2VsZi5vayA9IEZhbHNlICAjIG5pZ2R5IG5pZSByeXp5a3VqIGdyeSBixYLEmWRlbSBvY2VuaWFqxIVjZWdvCiAgICAgICAgICAgIHJldHVybiBOb25lCg=="))
(agent_dir / "knobs.py").write_bytes(base64.b64decode("IiIiUmVqZXN0ciBLTk9CUyAodjYuOCkg4oCUIGV3b2x1b3dhbG5hIHBvd2llcnpjaG5pYSBwYXJhbWV0csOzdyBwb2xpdHlraS4KCld6b3J6ZWMgeiBwcm9qZWt0dSBQb2tlbW9uIChjODdmN2E0OCk6IHdzenlzdGtpZSBtYWdpY3puZSBsaWN6YnkgYWdlbnRhIHcgamVkbnltCnPFgm93bmlrdSBgUFRDR18qYCDihpIgZXdvbHVjamEgQ0VNIHBvIDI2IHBva3LEmXTFgmFjaCAoMTAwIG11dGFudMOzdywgMTM4IGdpZXIvbXV0YW50KQp6ZGHFgmEgK3BrdCB0YW0sIGdkemllIHDFgnl0a2llIE1MIGRhxYJvIMKxMC4gVSBuYXMgYW5hbG9naWN6bmllOiBBUkMzXyoga25vYnMuCgogbmFkcGlzYW5pZTogZW52IGBBUkMzX0tOT0JTX0pTT05gID0gSlNPTiB6IFBPRFpCSU9SRU0ga2x1Y3p5IChyZXN6dGEgeiBkb215xZtsbnljaCkuCkpTT04gamVzdCBjenl0YW55IFJBWiBwcnp5IGltcG9yY2llIG1vZHXFgnU7IHRlc3R5IHVzdGF3aWFqxIUgd3Bpc3kgd3Byb3N0LgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBqc29uCmltcG9ydCBvcwoKS05PQlM6IGRpY3Rbc3RyLCBmbG9hdF0gPSB7CiAgICAjIGF1ZHl0IHV0a25pxJljaWEg4oCUIHd5endhbGFjeiBha2N5am55CiAgICAiQVVESVRfQUNUSU9OUyI6IDQwLCAgICAgICAgICAgICMgcHLDs2cgYWtjamkgbmEgcG96aW9taWUKICAgICJBVURJVF9BQ1RJT05TX1JFQVJNIjogMTUsICAgICAgIyBrb2xlam55IGF1ZHl0IHBvIHR5bHUgYWtjamFjaAogICAgIyBhdWR5dCB1dGtuacSZY2lhIOKAlCB3eXp3YWxhY3ogdG9rZW5vd3kKICAgICJBVURJVF9UT0tFTlMiOiA1MF8wMDAsICAgICAgICAgIyBwcsOzZyB0b2tlbsOzdyBuYSBwb3ppb21pZQogICAgIkFVRElUX1RPS0VOU19SRUFSTSI6IDIwXzAwMCwgICAjIGtvbGVqbnkgYXVkeXQgcG8gdHlsdSB0b2tlbmFjaAogICAgIkFVRElUX0VTQ0FMQVRFX0ZST00iOiAyLCAgICAgICAjIG9kIHRlZ28gbnVtZXJ1IGF1ZHl0dSBkb2Nob2R6aSBTVFJBVEVHWSBSRVNFVAogICAgIyBhZGFwdGFjeWpuYSB0ZW1wZXJhdHVyYQogICAgIlRFTVBfQkFTRSI6IDAuNiwgICAgICAgICAgICAgICAjIChrb21wYXR5Ymlsbm/Fm8SHIHogY29uZmlnaWVtOyBjaGF0IHXFvHl3YSBjb25maWd1IGdkeSBOb25lKQogICAgIlRFTVBfRlJFU0giOiAwLjcsICAgICAgICAgICAgICAjIMWbd2llxbx5IHBvemlvbSAoPCBGUkVTSF9BQ1RJT05TIGFrY2ppKQogICAgIlRFTVBfRlJFU0hfQUNUSU9OUyI6IDUsCiAgICAiVEVNUF9TVFVDSyI6IDAuOSwgICAgICAgICAgICAgICMgcG8gZXNrYWxhY2ppIChhdWR5dCA+PSBBVURJVF9FU0NBTEFURV9GUk9NKQogICAgIyBwcm9iZSBvcGVuaW5nCiAgICAiUFJPQkVfTUFYIjogNCwgICAgICAgICAgICAgICAgICMgbWFrcy4gYWtjamkta2llcnVua8OzdyBuYSBzdGFydCBwb3ppb211CiAgICAjIGJhdGNoaW5nCiAgICAiUFJPTVBUX0JBVENIX01BWCI6IDgsICAgICAgICAgICMgaWxlIGFrY2ppIHBsYW5vd2HEhyB3IGplZG5laiBvZHBvd2llZHppIChwcm9tcHQpCiAgICAiUVVFVUVfTUFYIjogMTUsICAgICAgICAgICAgICAgICMgcmUtcGxhbiBwbyB0eWx1IGFrY2phY2ggeiBrb2xlamtpCiAgICAjIHBhbWnEmcSHCiAgICAiTk9URVNfTUFYIjogNCwgICAgICAgICAgICAgICAgICMgbm90YXRraSDigJ5ub3RlcyIgdHJ6eW1hbmUgamVkbm9jemXFm25pZQogICAgIkxFU1NPTlNfTUFYIjogOCwgICAgICAgICAgICAgICAjIGxla2NqZSBtacSZZHp5IGdyYW1pIHcgcnVuaWUKICAgICJMRVZFTF9OT1RFU19LRUVQIjogMywgICAgICAgICAgIyBub3RhdGtpIHJvendpxIV6YW55Y2ggcG96aW9tw7N3CiAgICAiTEVWRUxfTk9URVNfQ0hBUlMiOiA0MDAsCiAgICAiR0FNRV9PVkVSU19LRUVQIjogNSwKICAgICJHQU1FX09WRVJTX1NIT1ciOiAzLAogICAgIyBuby1vcCBkZXRlY3RvcgogICAgIk5PT1BfV0lORE9XIjogMTAsCiAgICAiTk9PUF9NSU5fU0FNUExFUyI6IDYsCiAgICAiTk9PUF9UUklHR0VSIjogMywKICAgICMgdjYuOTogcnVudGltZSBkZXRla2NqYSBrbGFzeSBjZWx1IChjb3VudGVyLXBpY2tpbmcgeiBpY2ggdjcuMCkKICAgICJHT0FMX0NMQVNTIjogMSwgICAgICAgICAgICAgICAgIyAxID0gd3lrcnlqIGtsYXPEmSB6IDEuIHBsYW5zenkgaSB3c3RyenlrbmlqIGRvIHByb21wdMOzdwogICAgIyB2Ni45OiBkZXRlcm1pbmlzdHljem55IGZlZWRiYWNrIHdhcnRvxZtjaSBzdGFudSAoYmV6IE1MKSB3IHVzZXIgdHVybgogICAgIlZBTFVFX0dBVEUiOiAwLCAgICAgICAgICAgICAgICAjIDEgPSBsaW5pYSDigJ5EZXRlcm1pbmlzdGljIHZhbHVlIGVzdGltYXRlIiB3IHByb21wY2llCn0KCgpkZWYgYXBwbHlfb3ZlcnJpZGVzKHJhdzogc3RyIHwgTm9uZSkgLT4gTm9uZToKICAgICIiIk5hZHBpc3ogS05PQlMgeiBKU09OIChjesSZxZtjaW93eSkuIE5pZXpuYW5lIGtsdWN6ZSBwb21pamFuZSB6IG9zdHJ6ZcW8ZW5pZW0uIiIiCiAgICBpZiBub3QgcmF3OgogICAgICAgIHJldHVybgogICAgdHJ5OgogICAgICAgIGJsb2IgPSBqc29uLmxvYWRzKHJhdykKICAgIGV4Y2VwdCAoanNvbi5KU09ORGVjb2RlRXJyb3IsIFR5cGVFcnJvcik6CiAgICAgICAgcHJpbnQoZiJbS05PQlNdIEFSQzNfS05PQlNfSlNPTiBuaWUgamVzdCBwb3ByYXdueW0gSlNPTiDigJQgcG9taWphbToge3Jhd1s6ODBdfSIpCiAgICAgICAgcmV0dXJuCiAgICBpZiBub3QgaXNpbnN0YW5jZShibG9iLCBkaWN0KToKICAgICAgICBwcmludCgiW0tOT0JTXSBBUkMzX0tOT0JTX0pTT04gbmllIGplc3Qgb2JpZWt0ZW0g4oCUIHBvbWlqYW0iKQogICAgICAgIHJldHVybgogICAgZm9yIGtleSwgdmFsdWUgaW4gYmxvYi5pdGVtcygpOgogICAgICAgIGlmIGtleSBpbiBLTk9CUyBhbmQgaXNpbnN0YW5jZSh2YWx1ZSwgKGludCwgZmxvYXQpKSBhbmQgbm90IGlzaW5zdGFuY2UodmFsdWUsIGJvb2wpOgogICAgICAgICAgICBLTk9CU1trZXldID0gdHlwZShLTk9CU1trZXldKSh2YWx1ZSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBwcmludChmIltLTk9CU10gcG9taWphbSBuaWV6bmFuZS96xYJlIHBvbGU6IHtrZXl9PXt2YWx1ZSFyfSIpCgoKIyBTbmFwc2hvdCBkb215xZtsbnljaCB3YXJ0b8WbY2kgKGRvIHJlc2V0dSB3IHdvcmtlcmFjaCBtdWx0aXByb2Nlc3NpbmcpLgpERUZBVUxUUzogZGljdFtzdHIsIGZsb2F0XSA9IGRpY3QoS05PQlMpCgoKZGVmIHJlc2V0X3RvX2RlZmF1bHRzKCkgLT4gTm9uZToKICAgICIiIlByenl3csOzxIcgd2FydG/Fm2NpIGRvbXnFm2xuZSAod29ya2VyIG11bHRpcHJvY2Vzc2luZyBwcnplZCB3xYJhc255bSBuYWRwaXNhbmllbSkuIiIiCiAgICBLTk9CUy5jbGVhcigpCiAgICBLTk9CUy51cGRhdGUoREVGQVVMVFMpCgoKYXBwbHlfb3ZlcnJpZGVzKG9zLmVudmlyb24uZ2V0KCJBUkMzX0tOT0JTX0pTT04iKSkK"))
(agent_dir / "goal_classifier.py").write_bytes(base64.b64decode("IiIidjYuOTogUnVudGltZSBnb2FsLWNsYXNzIGRldGVjdGlvbiAoY291bnRlci1waWNraW5nKSDigJQgcG9ydCB6IGljaCB2Ny4wCihQb2tlbW9uL2FyZW5hLzc5MTFmOTUwLXBva2Vtb24gQDNhZDZhNDcsIGdvYWxfY2xhc3NpZmllci5weSksIHogcG9wcmF3a2FtaS4KCktsYXN5ICh0YXhvbm9taWEgR09BTF9DQVRBTE9HL0ZJRUxEIEdVSURFKToKQ09WRVIsIFJFQUNILCBDT1BZLCBSRURVQ0UsIFBMQUNFLCBGTE9XLCBBTElHTiwgTE9DQUxfQ09OU1RSQUlOVFMsIFVOS05PV04uCgpSw7PFvG5pY2UgdnMgaWNoIHdlcnNqYSAodWN6Y2l3aWUpOgoxLiBCVUdGSVg6IHB1c3RhL3TFgm8tb25seSBwbGFuc3phIGRhd2HFgmEgdSBuaWNoIENPUFkgKHd6b3J6ZWMgMTDDlzEwIHogemVyw7N3CiAgIOKAnnBva3J5d2HFgiIgaW5uZSB6ZXJvd2Ugb2tuYSB6IHNpbWlsYXJpdHkgMS4wKS4gVSBuYXM6IGVhcmx5LXJldHVybiBVTktOT1dOCiAgIGRsYSBwdXN0eWNoIHBsYW5zeiArIHd6b3J6ZWMgbmFyb8W8bmlrYSBtdXNpIG1pZcSHIOKJpTQga29tw7Nya2kg4omgIDAuCjIuIEJVR0ZJWDogbWV0YWRhdGEgZ3JpZHUgbGljem9uYSB6IGxpc3RbbGlzdFtpbnRdXSDigJQgdSBuYXMgcHJ6eWptdWplbXkgdGXFvAogICB0dXBsZVt0dXBsZVtpbnQsIC4uLl0sIC4uLl0gKEZvcm1hdCBGcmFtZUluZm8uZ3JpZCksIG5vcm1hbGl6YWNqYSBuYSB3ZWrFm2NpdS4KMy4gTklFIHBvcnR1amVteSBTdHJhdGVneVBhcmFtcyAod2FnaSBoZXVyeXN0eWtpIGRvIGNvdW50ZXItcGlja2luZ3UpIOKAlCB1IG5pY2gKICAgdG8gbWFydHd5IGtvZCAobmlrdCBuaWUgaW1wb3J0dWplKSwgdSBuYXMgaGV1cnlzdHljem55Y2ggd2FnIG5pZSBtYTsgbmFzegogICBjb3VudGVyLXBpY2tpbmcgZHppYcWCYSBuYSBwb3ppb21pZSBwcm9tcHR1OiBTVFJBVEVHWV9ISU5UUyBwb25pxbxlai4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCgpHT0FMX0NMQVNTRVMgPSAoCiAgICAiQ09WRVIiLCAiUkVBQ0giLCAiQ09QWSIsICJSRURVQ0UiLCAiUExBQ0UiLCAiRkxPVyIsICJBTElHTiIsCiAgICAiTE9DQUxfQ09OU1RSQUlOVFMiLCAiVU5LTk9XTiIsCikKCgpAZGF0YWNsYXNzCmNsYXNzIEdhbWVGZWF0dXJlczoKICAgICIiIkNlY2h5IHBsYW5zenkgdcW8eXdhbmUgZG8gZGV0ZWtjamkga2xhc3kuIiIiCiAgICAjIFBva3J5Y2llCiAgICBzbWFsbF9tYXJrZXJzX2NvdW50OiBpbnQgPSAwICAgICAgICAgIyBJem9sb3dhbmUga29tw7Nya2kgKG1hxYJlIG1hcmtlcnkpCiAgICBvdXRsaW5lZF9wb3NpdGlvbnNfY291bnQ6IGludCA9IDAgICAgIyBLb23Ds3JraSDigJ5vYnJ5c293YW5lIiBpbm55bSBrb2xvcmVtCiAgICBjb2xvcmVkX2VkZ2VfbWFya2Vyc19jb3VudDogaW50ID0gMCAgIyBOaWV6ZXJvd2Uga29tw7Nya2kgbmEga3Jhd8SZZHppCiAgICAjIFd6b3JjZQogICAgcGF0dGVybl9pbl9jb3JuZXI6IGJvb2wgPSBGYWxzZSAgICAgICMgUG93dGFyemFsbnkgd3pvcnplYyAxMMOXMTAgeiByb2d1CiAgICBwYXR0ZXJuX3NpbWlsYXJpdHk6IGZsb2F0ID0gMC4wCiAgICAjIE9iaWVrdHkKICAgIHBpZWNlc19jb3VudDogaW50ID0gMCAgICAgICAgICAgICAgICAjIE1hxYJlIG9iaWVrdHkgKDHigJM0IGtvbcOzcmtpKQogICAgZW1wdHlfY2VsbHNfY291bnQ6IGludCA9IDAKICAgIGJveGVzX2NvdW50OiBpbnQgPSAwICAgICAgICAgICAgICAgICAjIER1xbxlIG9iaWVrdHkgKOKJpTUga29tw7NyZWspCiAgICB0YXJnZXRfem9uZV9kZXRlY3RlZDogYm9vbCA9IEZhbHNlCiAgICAjIE1lY2hhbmlraQogICAgZGlzcGVuc2VyX2RldGVjdGVkOiBib29sID0gRmFsc2UgICAgICMgSmVkbm9saXR5IG5pZXplcm93eSB3aWVyc3ogMCAoRkxPVykKICAgIHZlc3NlbHNfY291bnQ6IGludCA9IDAKICAgIHRpbGVfa2V5c18zeDNfY291bnQ6IGludCA9IDAgICAgICAgICAjIEfEmXN0ZSBibG9raSAzw5czIChMT0NBTF9DT05TVFJBSU5UUykKICAgICMgR3JpZAogICAgZ3JpZF9oZWlnaHQ6IGludCA9IDAKICAgIGdyaWRfd2lkdGg6IGludCA9IDAKICAgIHVuaXF1ZV9jb2xvcnM6IGludCA9IDAKICAgIG5vbl96ZXJvX2NlbGxzOiBpbnQgPSAwCgoKZGVmIGV4dHJhY3RfZ2FtZV9mZWF0dXJlcyhncmlkKSAtPiBHYW1lRmVhdHVyZXM6CiAgICAiIiJFa3N0cmFrY2phIGNlY2ggeiBncmlkdSAobGlzdHkgbHViIGtyb3RraSB3aWVyc3p5OyAwID0gdMWCbykuIiIiCiAgICByb3dzID0gW2xpc3QocikgZm9yIHIgaW4gZ3JpZF0KICAgIGlmIG5vdCByb3dzIG9yIG5vdCByb3dzWzBdOgogICAgICAgIHJldHVybiBHYW1lRmVhdHVyZXMoKQogICAgaGVpZ2h0ID0gbGVuKHJvd3MpCiAgICB3aWR0aCA9IGxlbihyb3dzWzBdKQoKICAgIGZsYXQgPSBbY2VsbCBmb3Igcm93IGluIHJvd3MgZm9yIGNlbGwgaW4gcm93XQogICAgdW5pcXVlX2NvbG9ycyA9IGxlbihzZXQoZmxhdCkpCiAgICBub25femVyb19jZWxscyA9IHN1bSgxIGZvciBjZWxsIGluIGZsYXQgaWYgY2VsbCAhPSAwKQogICAgZW1wdHlfY2VsbHMgPSBzdW0oMSBmb3IgY2VsbCBpbiBmbGF0IGlmIGNlbGwgPT0gMCkKCiAgICAjIFd6b3J6ZWMgdyByb2d1IChDT1BZKTogbGV3eSBnw7NybnkgMTDDlzEwIHZzIGlubmUgb2tuYSAxMMOXMTAgKGtyb2sgNSkuCiAgICAjIEJVR0ZJWCAjMyB2cyBpY2ggd2Vyc2phOiBvbmkgbGljenlsaSB0ZcW8IGRvcGFzb3dhbmlhIHplcm8tdnMtemVybywgd2nEmWMKICAgICMgbmEgcnphZGtpY2ggcGxhbnN6YWNoIG5pZW1hbCB3c3p5c3RrbyBkYXdhxYJvIHNpbWlsYXJpdHkgfjAuOSsg4oaSIGZhxYJzenl3ZQogICAgIyBDT1BZLiBVIG5hcyBzY29yZSBsaWN6eSBXWcWBxIRDWk5JRSBzdHJ1a3R1csSZIG5pZXplcm93xIUgKHRyYWZpZW5pYSB3CiAgICAjIG5pZXplcm93ZSBrb23Ds3JraSB3em9yY2EgLyAodHJhZmllbmlhICsgbmFkbWlhcm93ZSBuaWV6ZXJvd2Ugb2tuYSkpLgogICAgcGF0dGVybl9pbl9jb3JuZXIgPSBGYWxzZQogICAgcGF0dGVybl9zaW1pbGFyaXR5ID0gMC4wCiAgICBpZiBoZWlnaHQgPj0gMTAgYW5kIHdpZHRoID49IDEwOgogICAgICAgIHBhdHRlcm4gPSBbcm93WzoxMF0gZm9yIHJvdyBpbiByb3dzWzoxMF1dCiAgICAgICAgcGF0X256ID0gWyhyLCBjKSBmb3IgciBpbiByYW5nZSgxMCkgZm9yIGMgaW4gcmFuZ2UoMTApIGlmIHBhdHRlcm5bcl1bY10gIT0gMF0KICAgICAgICBwYXRfY29sb3JzID0ge3BhdHRlcm5bcl1bY10gZm9yIChyLCBjKSBpbiBwYXRfbnp9CiAgICAgICAgIyBCVUdGSVggIzUgKDE1LjEwLCBsZWtjamEgeiBpY2ggdjcuMCk6IHVuaWZvcm0gcsOzZyAoMSBrb2xvcikg4oCecGFzdWplIgogICAgICAgICMgZG8gaW5ueWNoIHVuaWZvcm0gb2tpZW4g4oaSIGZhxYJzenl3ZSBDT1BZIG5hIHBsYW5zemFjaCBGTE9XL3BlxYJueWNoLgogICAgICAgICMgUHJhd2R6aXd5IHBvd3RhcnphbG55IHd6b3J6ZWMgbWEgc3RydWt0dXLEmTogPj0yIGtvbG9yeS4KICAgICAgICBpZiBsZW4ocGF0X256KSA+PSA0IGFuZCBsZW4ocGF0X2NvbG9ycykgPj0gMjoKICAgICAgICAgICAgYmVzdCA9IDAuMAogICAgICAgICAgICBmb3IgciBpbiByYW5nZSgwLCBoZWlnaHQgLSA5LCA1KToKICAgICAgICAgICAgICAgIGZvciBjIGluIHJhbmdlKDAsIHdpZHRoIC0gOSwgNSk6CiAgICAgICAgICAgICAgICAgICAgaWYgciA9PSAwIGFuZCBjID09IDA6CiAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICAgICAgd2luZG93X256ID0gc3VtKAogICAgICAgICAgICAgICAgICAgICAgICAxIGZvciByciBpbiByYW5nZSgxMCkgZm9yIGNjIGluIHJhbmdlKDEwKQogICAgICAgICAgICAgICAgICAgICAgICBpZiByICsgcnIgPCBoZWlnaHQgYW5kIGMgKyBjYyA8IHdpZHRoCiAgICAgICAgICAgICAgICAgICAgICAgIGFuZCByb3dzW3IgKyBycl1bYyArIGNjXSAhPSAwKQogICAgICAgICAgICAgICAgICAgIG1hdGNoZXMgPSBzdW0oCiAgICAgICAgICAgICAgICAgICAgICAgIDEgZm9yIChyciwgY2MpIGluIHBhdF9uegogICAgICAgICAgICAgICAgICAgICAgICBpZiByICsgcnIgPCBoZWlnaHQgYW5kIGMgKyBjYyA8IHdpZHRoCiAgICAgICAgICAgICAgICAgICAgICAgIGFuZCByb3dzW3IgKyBycl1bYyArIGNjXSA9PSBwYXR0ZXJuW3JyXVtjY10pCiAgICAgICAgICAgICAgICAgICAgZXh0cmEgPSBtYXgoMCwgd2luZG93X256IC0gbWF0Y2hlcykKICAgICAgICAgICAgICAgICAgICBzY29yZSA9IG1hdGNoZXMgLyAobWF0Y2hlcyArIGV4dHJhKSBpZiAobWF0Y2hlcyArIGV4dHJhKSBlbHNlIDAuMAogICAgICAgICAgICAgICAgICAgIGJlc3QgPSBtYXgoYmVzdCwgc2NvcmUpCiAgICAgICAgICAgIHBhdHRlcm5fc2ltaWxhcml0eSA9IGJlc3QKICAgICAgICAgICAgcGF0dGVybl9pbl9jb3JuZXIgPSBiZXN0ID4gMC43CgogICAgIyBNYcWCZSBtYXJrZXJ5IChDT1ZFUik6IGtvbcOzcmthLCBrdMOzcmVqIDQtc8SFc2llZHppIG5pZSBtYWrEhSBqZWoga29sb3J1LgogICAgc21hbGxfbWFya2VycyA9IDAKICAgIGZvciByIGluIHJhbmdlKGhlaWdodCk6CiAgICAgICAgZm9yIGMgaW4gcmFuZ2Uod2lkdGgpOgogICAgICAgICAgICBpZiByb3dzW3JdW2NdID09IDA6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBuZWlnaGJvcnMgPSBbXQogICAgICAgICAgICBmb3IgZHIsIGRjIGluICgoLTEsIDApLCAoMSwgMCksICgwLCAtMSksICgwLCAxKSk6CiAgICAgICAgICAgICAgICBuciwgbmMgPSByICsgZHIsIGMgKyBkYwogICAgICAgICAgICAgICAgaWYgMCA8PSBuciA8IGhlaWdodCBhbmQgMCA8PSBuYyA8IHdpZHRoOgogICAgICAgICAgICAgICAgICAgIG5laWdoYm9ycy5hcHBlbmQocm93c1tucl1bbmNdKQogICAgICAgICAgICBpZiByb3dzW3JdW2NdIG5vdCBpbiBuZWlnaGJvcnM6CiAgICAgICAgICAgICAgICBzbWFsbF9tYXJrZXJzICs9IDEKCiAgICAjIE9icnlzb3dhbmUgcG96eWNqZSAoUkVBQ0gpOiBrb23Ds3JrYSDiiaAgMCwgd29rw7PFgiDiiaUyIHLDs8W8bmUga29sb3J5LCB3IHR5bQogICAgIyDFvGFkZW4gcsOzd255IGplaiBrb2xvcm93aS4KICAgIG91dGxpbmVkX3Bvc2l0aW9ucyA9IDAKICAgIGZvciByIGluIHJhbmdlKGhlaWdodCk6CiAgICAgICAgZm9yIGMgaW4gcmFuZ2Uod2lkdGgpOgogICAgICAgICAgICBpZiByb3dzW3JdW2NdID09IDA6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBuZWlnaGJvcl9jb2xvcnMgPSBzZXQoKQogICAgICAgICAgICBmb3IgZHIsIGRjIGluICgoLTEsIDApLCAoMSwgMCksICgwLCAtMSksICgwLCAxKSk6CiAgICAgICAgICAgICAgICBuciwgbmMgPSByICsgZHIsIGMgKyBkYwogICAgICAgICAgICAgICAgaWYgMCA8PSBuciA8IGhlaWdodCBhbmQgMCA8PSBuYyA8IHdpZHRoOgogICAgICAgICAgICAgICAgICAgIG5laWdoYm9yX2NvbG9ycy5hZGQocm93c1tucl1bbmNdKQogICAgICAgICAgICBpZiBsZW4obmVpZ2hib3JfY29sb3JzKSA+IDEgYW5kIHJvd3Nbcl1bY10gbm90IGluIG5laWdoYm9yX2NvbG9yczoKICAgICAgICAgICAgICAgIG91dGxpbmVkX3Bvc2l0aW9ucyArPSAxCgogICAgIyBLcmF3xJlkemlvd2UgbWFya2VyeSAoQUxJR04pCiAgICBjb2xvcmVkX2VkZ2VfbWFya2VycyA9IDAKICAgIGZvciByIGluIHJhbmdlKGhlaWdodCk6CiAgICAgICAgZm9yIGMgaW4gcmFuZ2Uod2lkdGgpOgogICAgICAgICAgICBpZiByIGluICgwLCBoZWlnaHQgLSAxKSBvciBjIGluICgwLCB3aWR0aCAtIDEpOgogICAgICAgICAgICAgICAgaWYgcm93c1tyXVtjXSAhPSAwOgogICAgICAgICAgICAgICAgICAgIGNvbG9yZWRfZWRnZV9tYXJrZXJzICs9IDEKCiAgICBkZWYgX2NvbXBvbmVudHMobWluX3NpemU6IGludCwgbWF4X3NpemU6IGludCB8IE5vbmUpIC0+IGludDoKICAgICAgICAiIiJMaWN6IHNww7NqbnljaCBvYnN6YXLDs3cgdGVnbyBzYW1lZ28ga29sb3J1IG8gcm96bWlhcnplIHcgZ3JhbmljYWNoLiIiIgogICAgICAgIHNlZW4gPSBbW0ZhbHNlXSAqIHdpZHRoIGZvciBfIGluIHJhbmdlKGhlaWdodCldCiAgICAgICAgY291bnQgPSAwCiAgICAgICAgZm9yIHIgaW4gcmFuZ2UoaGVpZ2h0KToKICAgICAgICAgICAgZm9yIGMgaW4gcmFuZ2Uod2lkdGgpOgogICAgICAgICAgICAgICAgaWYgc2VlbltyXVtjXSBvciByb3dzW3JdW2NdID09IDA6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHF1ZXVlID0gWyhyLCBjKV0KICAgICAgICAgICAgICAgIHNpemUgPSAwCiAgICAgICAgICAgICAgICBjb2xvciA9IHJvd3Nbcl1bY10KICAgICAgICAgICAgICAgIHdoaWxlIHF1ZXVlOgogICAgICAgICAgICAgICAgICAgIGNyLCBjYyA9IHF1ZXVlLnBvcCgpCiAgICAgICAgICAgICAgICAgICAgaWYgY3IgPCAwIG9yIGNjIDwgMCBvciBjciA+PSBoZWlnaHQgb3IgY2MgPj0gd2lkdGg6CiAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICAgICAgaWYgc2Vlbltjcl1bY2NdIG9yIHJvd3NbY3JdW2NjXSAhPSBjb2xvcjoKICAgICAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgICAgICBzZWVuW2NyXVtjY10gPSBUcnVlCiAgICAgICAgICAgICAgICAgICAgc2l6ZSArPSAxCiAgICAgICAgICAgICAgICAgICAgZm9yIGRyLCBkYyBpbiAoKC0xLCAwKSwgKDEsIDApLCAoMCwgLTEpLCAoMCwgMSkpOgogICAgICAgICAgICAgICAgICAgICAgICBxdWV1ZS5hcHBlbmQoKGNyICsgZHIsIGNjICsgZGMpKQogICAgICAgICAgICAgICAgaWYgc2l6ZSA+PSBtaW5fc2l6ZSBhbmQgKG1heF9zaXplIGlzIE5vbmUgb3Igc2l6ZSA8PSBtYXhfc2l6ZSk6CiAgICAgICAgICAgICAgICAgICAgY291bnQgKz0gMQogICAgICAgIHJldHVybiBjb3VudAoKICAgIGJveGVzID0gX2NvbXBvbmVudHMoNSwgTm9uZSkgICAgICAgICAgIyBTa3J6eW5raSAoUExBQ0UpOiBkdcW8ZSBvYnN6YXJ5CiAgICBwaWVjZXMgPSBfY29tcG9uZW50cygxLCA0KSAgICAgICAgICAgICMgUGlvbmtpIChSRURVQ0UpOiBtYcWCZSBvYmlla3R5CgogICAgIyBTdHJlZmEgY2VsdSAoUExBQ0UsIHVwcm9zemN6b25lKTogZ8SZc3RhIHBsYW5zemEgeiBkdcW8eW1pIG9iaWVrdGFtaS4KICAgIHRhcmdldF96b25lID0gYm94ZXMgPiAwIGFuZCBub25femVyb19jZWxscyA+IGhlaWdodCAqIHdpZHRoICogMC41CgogICAgIyBEb3pvd25payAoRkxPVyk6IHdpZXJzeiAwIGplZG5vbGl0eSBuaWV6ZXJvd3kgKHN6ZXJva2/Fm8SHID4gNSkuCiAgICBkaXNwZW5zZXIgPSBGYWxzZQogICAgdG9wX2NvbG9ycyA9IHtjIGZvciBjIGluIHJvd3NbMF0gaWYgYyAhPSAwfQogICAgaWYgbGVuKHRvcF9jb2xvcnMpID09IDEgYW5kIHdpZHRoID4gNToKICAgICAgICBkaXNwZW5zZXIgPSBUcnVlCgogICAgIyBOYWN6eW5pYSAoRkxPVywgdXByb3N6Y3pvbmUpOiBuaWV6ZXJvd2Uga29tw7Nya2kgdyA1IGRvbG55Y2ggd2llcnN6YWNoLgogICAgdmVzc2VscyA9IDAKICAgIGlmIGhlaWdodCA+IDU6CiAgICAgICAgYm90dG9tX25vbnplcm8gPSBzdW0oMSBmb3Igcm93IGluIHJvd3NbLTU6XSBmb3IgY2VsbCBpbiByb3cgaWYgY2VsbCAhPSAwKQogICAgICAgIHZlc3NlbHMgPSBtYXgoMCwgYm90dG9tX25vbnplcm8gLy8gMTApCgogICAgIyBLYWZlbGtpLWtsdWN6ZSAzw5czIChMT0NBTF9DT05TVFJBSU5UUyk6IMWbcm9ka2kgdyBwZcWCbmkgbmllemVyb3d5Y2ggM8OXMy4KICAgIHRpbGVfa2V5cyA9IDAKICAgIGZvciByIGluIHJhbmdlKDEsIGhlaWdodCAtIDEpOgogICAgICAgIGZvciBjIGluIHJhbmdlKDEsIHdpZHRoIC0gMSk6CiAgICAgICAgICAgIGlmIHJvd3Nbcl1bY10gPT0gMDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGRlbnNlID0gVHJ1ZQogICAgICAgICAgICBmb3IgZHIgaW4gKC0xLCAwLCAxKToKICAgICAgICAgICAgICAgIGZvciBkYyBpbiAoLTEsIDAsIDEpOgogICAgICAgICAgICAgICAgICAgIGlmIHJvd3NbciArIGRyXVtjICsgZGNdID09IDA6CiAgICAgICAgICAgICAgICAgICAgICAgIGRlbnNlID0gRmFsc2UKICAgICAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIG5vdCBkZW5zZToKICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBpZiBkZW5zZToKICAgICAgICAgICAgICAgIHRpbGVfa2V5cyArPSAxCgogICAgcmV0dXJuIEdhbWVGZWF0dXJlcygKICAgICAgICBzbWFsbF9tYXJrZXJzX2NvdW50PXNtYWxsX21hcmtlcnMsCiAgICAgICAgb3V0bGluZWRfcG9zaXRpb25zX2NvdW50PW91dGxpbmVkX3Bvc2l0aW9ucywKICAgICAgICBjb2xvcmVkX2VkZ2VfbWFya2Vyc19jb3VudD1jb2xvcmVkX2VkZ2VfbWFya2VycywKICAgICAgICBwYXR0ZXJuX2luX2Nvcm5lcj1wYXR0ZXJuX2luX2Nvcm5lciwKICAgICAgICBwYXR0ZXJuX3NpbWlsYXJpdHk9cGF0dGVybl9zaW1pbGFyaXR5LAogICAgICAgIHBpZWNlc19jb3VudD1waWVjZXMsCiAgICAgICAgZW1wdHlfY2VsbHNfY291bnQ9ZW1wdHlfY2VsbHMsCiAgICAgICAgYm94ZXNfY291bnQ9Ym94ZXMsCiAgICAgICAgdGFyZ2V0X3pvbmVfZGV0ZWN0ZWQ9dGFyZ2V0X3pvbmUsCiAgICAgICAgZGlzcGVuc2VyX2RldGVjdGVkPWRpc3BlbnNlciwKICAgICAgICB2ZXNzZWxzX2NvdW50PXZlc3NlbHMsCiAgICAgICAgdGlsZV9rZXlzXzN4M19jb3VudD10aWxlX2tleXMsCiAgICAgICAgZ3JpZF9oZWlnaHQ9aGVpZ2h0LAogICAgICAgIGdyaWRfd2lkdGg9d2lkdGgsCiAgICAgICAgdW5pcXVlX2NvbG9ycz11bmlxdWVfY29sb3JzLAogICAgICAgIG5vbl96ZXJvX2NlbGxzPW5vbl96ZXJvX2NlbGxzLAogICAgKQoKCmRlZiBkZXRlY3RfZ29hbF9jbGFzcyhncmlkKSAtPiBzdHI6CiAgICAiIiJEZXRla2NqYSBrbGFzeSBncnkgeiBwaWVyd3N6ZWogcGxhbnN6eS4gWndyYWNhIGVsZW1lbnQgR09BTF9DTEFTU0VTLiIiIgogICAgcm93cyA9IFtsaXN0KHIpIGZvciByIGluIGdyaWRdCiAgICBpZiBub3Qgcm93cyBvciBub3Qgcm93c1swXToKICAgICAgICByZXR1cm4gIlVOS05PV04iCiAgICBmZWF0dXJlcyA9IGV4dHJhY3RfZ2FtZV9mZWF0dXJlcyhyb3dzKQogICAgIyBCVUdGSVggdnMgaWNoIHdlcnNqYTogcHVzdGEvdMWCby1vbmx5IHBsYW5zemEg4oaSIFVOS05PV04gKHUgbmljaCBkYXdhxYJhCiAgICAjIGZhxYJzenl3ZSBDT1BZIHByemV6IOKAnnd6b3J6ZWMiIHogc2FteWNoIHplcikuCiAgICBpZiBmZWF0dXJlcy5ub25femVyb19jZWxscyA9PSAwOgogICAgICAgIHJldHVybiAiVU5LTk9XTiIKICAgICMgUHJpb3J5dGV0OiBvZCBuYWpiYXJkemllaiBzcGVjeWZpY3pueWNoLiBMZWtjamUgeiBpY2ggdjcuMCAoMTUuMTApOgogICAgIyAoYSkgZ8SZc3R5IHd6b3J6ZWMgQ09QWSBtYSBwZcWCbmUgYmxva2kgM8OXMyDihpIgTE9DQUxfQ09OU1RSQUlOVFMgKHRpbGVfa2V5cz4zKQogICAgIyBtdXN6xIUgY3pla2HEhyBuYSBDT1BZOyAoYikgdW5pZm9ybSBwbGFuc3phIHogZG96b3duaWtpZW0gbWEg4oCeZ8SZc3R5IHLDs2ciLAogICAgIyBhbGUgd3pvcnplYyBiZXogPj0yIGtvbG9yw7N3IG5pZSBsaWN6eSBzacSZIChCVUdGSVggIzUpIOKAlCBGTE9XIHoga29ua3JldG55bQogICAgIyBkb3pvd25pa2llbStuYWN6eW5pYW1pIHJvenN0cnp5Z2FteSBuYWpwaWVydy4KICAgIGlmIGZlYXR1cmVzLmRpc3BlbnNlcl9kZXRlY3RlZCBhbmQgZmVhdHVyZXMudmVzc2Vsc19jb3VudCA+IDI6CiAgICAgICAgcmV0dXJuICJGTE9XIgogICAgaWYgZmVhdHVyZXMucGF0dGVybl9pbl9jb3JuZXI6CiAgICAgICAgcmV0dXJuICJDT1BZIgogICAgaWYgZmVhdHVyZXMudGlsZV9rZXlzXzN4M19jb3VudCA+IDM6CiAgICAgICAgcmV0dXJuICJMT0NBTF9DT05TVFJBSU5UUyIKICAgIGlmIGZlYXR1cmVzLmJveGVzX2NvdW50ID4gMiBhbmQgZmVhdHVyZXMudGFyZ2V0X3pvbmVfZGV0ZWN0ZWQ6CiAgICAgICAgcmV0dXJuICJQTEFDRSIKICAgICMgQlVHRklYICM0IHZzIGljaCB3ZXJzamE6IHByw7NnIDIwIHphbWlhc3QgMTAg4oCUIHUgbmljaCBpem9sb3dhbmUgbWFya2VyeQogICAgIyBDT1ZFUiAoMS1rb23Ds3Jrb3dlID0g4oCecGlvbmtpIikgd3lncnl3YcWCeSBSRURVQ0UgcHJ6ZXoga29sZWpub8WbxIcgcmVndcWCLgogICAgaWYgZmVhdHVyZXMucGllY2VzX2NvdW50ID4gMjAgYW5kIGZlYXR1cmVzLmVtcHR5X2NlbGxzX2NvdW50ID4gZmVhdHVyZXMucGllY2VzX2NvdW50OgogICAgICAgIHJldHVybiAiUkVEVUNFIgogICAgaWYgZmVhdHVyZXMuY29sb3JlZF9lZGdlX21hcmtlcnNfY291bnQgPiA1OgogICAgICAgIHJldHVybiAiQUxJR04iCiAgICBpZiBmZWF0dXJlcy5vdXRsaW5lZF9wb3NpdGlvbnNfY291bnQgPiA1OgogICAgICAgIHJldHVybiAiUkVBQ0giCiAgICBpZiBmZWF0dXJlcy5zbWFsbF9tYXJrZXJzX2NvdW50ID4gMTA6CiAgICAgICAgcmV0dXJuICJDT1ZFUiIKICAgIHJldHVybiAiVU5LTk9XTiIKCgojIE5hc3ogY291bnRlci1waWNraW5nOiBwb3ppb20gcHJvbXB0dSAoaGludCB0YWt0eWN6bnkgcGVyIGtsYXNhKSwgemFtaWFzdCBpY2gKIyBTdHJhdGVneVBhcmFtcyAod2FnaSBoZXVyeXN0eWtpIOKAlCB1IG5hcyBtYXJ0d3kga29kLCBuaWUgbWFteSB0YWtpY2ggd2FnKS4KU1RSQVRFR1lfSElOVFM6IGRpY3Rbc3RyLCBzdHJdID0gewogICAgIkNPVkVSIjogInNtYWxsIG1hcmtlcnMgbGlrZWx5IG11c3QgQUxMIGJlIGNvdmVyZWQvYWN0aXZhdGVkIOKAlCBjb3VudCB0aGVtLCAiCiAgICAgICAgICAgICAidmVyaWZ5IGVhY2ggaW50ZXJhY3Rpb24gY2hhbmdlcyBleGFjdGx5IG9uZSBtYXJrZXIiLAogICAgIlJFQUNIIjogIm91dGxpbmVkIHBvc2l0aW9ucyBsaWtlbHkgbXVzdCBBTEwgYmUgdmlzaXRlZCDigJQgcGxhbiB0aGUgc2hvcnRlc3QgIgogICAgICAgICAgICAgInJvdXRlIGFuZCBjb25maXJtIGVhY2ggdmlzaXQgcmVnaXN0ZXJzIiwKICAgICJDT1BZIjogImEgcGF0dGVybiByZXBlYXRzIG9uIHRoZSBib2FyZCDigJQgcmVwcm9kdWNlIGl0IGVsc2V3aGVyZSAodHJhbnNsYXRpb24gIgogICAgICAgICAgICAib3IgY29tcGxldGlvbiksIGNvbXBhcmUgY2VsbC1ieS1jZWxsIiwKICAgICJSRURVQ0UiOiAibWFueSBzbWFsbCBwaWVjZXMgbXVzdCBiZSByZWR1Y2VkIHRvIGZldy9vbmUg4oCUIGZpbmQgdGhlIHJlbW92YWwgIgogICAgICAgICAgICAgICJydWxlIG9uIGEgc2luZ2xlIHBpZWNlIGZpcnN0IiwKICAgICJQTEFDRSI6ICJsYXJnZSBvYmplY3RzIGJlbG9uZyBpbnRvIGEgem9uZSDigJQgbW92ZSBvbmUgb2JqZWN0IGZ1bGx5IGZpcnN0IHRvICIKICAgICAgICAgICAgICJsZWFybiB0aGUgcGxhY2VtZW50IHJ1bGUiLAogICAgIkZMT1ciOiAiYSBkaXNwZW5zZXIvc291cmNlIGZlZWRzIHZlc3NlbHMg4oCUIHRyYWNlIHRoZSBmbG93IHBhdGggYmVmb3JlICIKICAgICAgICAgICAgInRvdWNoaW5nIGFueXRoaW5nIiwKICAgICJBTElHTiI6ICJlZGdlLWNvbG9yZWQgbWFya2VycyBzdWdnZXN0IG9yZGVyaW5nL2FsaWdubWVudCDigJQgdGVzdCB3aGV0aGVyICIKICAgICAgICAgICAgICJvcmRlciBtYXR0ZXJzIGJ5IGFsaWduaW5nIHR3byBtYXJrZXJzIGZpcnN0IiwKICAgICJMT0NBTF9DT05TVFJBSU5UUyI6ICJkZW5zZSAzw5czIGJsb2NrcyBoaW50IGF0IGxvY2FsLW5laWdoYm9yaG9vZCBydWxlcyDigJQgIgogICAgICAgICAgICAgICAgICAgICAgICAgImNoYW5nZSBvbmUgY2VsbCBhbmQgb2JzZXJ2ZSB3aGljaCBuZWlnaGJvcnMgcmVhY3QiLAp9Cg=="))

# Set environment for the agent
os.environ["LLM_BASE_URL"] = VLLM_BASE_URL
os.environ["LLM_MODEL_ID"] = SERVED_MODEL
os.environ["LLM_API_KEY"] = "EMPTY"
os.environ["LLM_CONTEXT_WINDOW"] = "32768"
os.environ["LLM_MAX_OUTPUT"] = "2048"
os.environ["LLM_TEMPERATURE"] = "0.6"
os.environ["LLM_TIMEOUT"] = "120"
os.environ["LLM_MULTIMODAL"] = "0"  # start without images to save tokens

print(f"Agent deployed in {agent_dir}")
print(f"LLM config: {SERVED_MODEL} @ {VLLM_BASE_URL}")

# --- Agent self-test (lesson from the Pokemon branches: dead code and silent
# except-pass killed whole runs there; our 14.10 parser bug was the same class).
# Fails the notebook HERE, before any GPU time is burned on a broken agent.
import sys as _sys
_sys.path.insert(0, "/tmp")
from arc_llm_agent import llm_policy as _lp

_src = (agent_dir / "llm_policy.py").read_text()
for _marker in ("_extract_action_objects", "FIELD GUIDE", "_probe_actions",
                "STRATEGY AUDIT", '"notes"', "LESSONS BETWEEN GAMES",
                "class ValueGate", "KNOBS", "GOAL_CLASS"):
    assert _marker in _src, f"stale payload: missing {_marker!r} in llm_policy.py"

_gc_src = (agent_dir / "goal_classifier.py").read_text()
assert "detect_goal_class" in _gc_src, "stale payload: goal_classifier.py"
assert _lp.detect_goal_class(((0, 0), (0, 0))) == "UNKNOWN", "goal classifier broken"

_fi = _lp.FrameInfo(grid=((0, 0), (0, 0)), step=0, level=0,
                    state_name="PLAYING", levels_completed=0,
                    available_actions=["ACTION1", "ACTION2", "ACTION6"])
_pol = _lp.LLMPolicy.__new__(_lp.LLMPolicy)
_acts = _pol._parse_response(
    '```json\n{"action": "UP", "reasoning": {"why": {"nested": 1}}}\n```', _fi)
assert _acts and _acts[0]["name"] == "ACTION1", "nested-JSON action was dropped!"
_probes = _lp.LLMPolicy._probe_actions(_fi)
assert [p["name"] for p in _probes] == ["ACTION1", "ACTION2"], "probe opening broken"
assert all(p["name"] != "ACTION6" for p in _probes), "probes must not click blindly"
print("ARC3_AGENT_SELFTEST_OK: parser + probe opening verified")


In [ ]:
import shutil
from pathlib import Path

if TRUE_SUBMISSION:
    # === COMPETITION RUN ===
    # Copy the ARC-AGI-3-Agents framework
    framework_src = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents")
    framework = WORKING / "ARC-AGI-3-Agents"
    if framework.exists():
        shutil.rmtree(framework)
    shutil.copytree(framework_src, framework)
    
    # Copy our agent into the templates directory
    target = framework / "agents" / "templates"
    target.mkdir(parents=True, exist_ok=True)
    for f in ("__init__.py", "grid_utils.py", "llm_policy.py", "llm_agent.py",
               "value_gate.py", "knobs.py", "goal_classifier.py"):
        shutil.copy2(Path("/tmp/arc_llm_agent") / f, target / f)
    
    # Register our agent
    init_content = """
\"\"\"Agent registry for ARC-AGI-3 LLM agent.\"\"\"
from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .recorder import Recorder
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
from .templates.llm_agent import LLMAgent

load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
    "llmagent": LLMAgent,
}
for recording in Recorder.list():
    AVAILABLE_AGENTS[recording] = Playback
"""
    (framework / "agents" / "__init__.py").write_text(init_content)
    
    # Write .env
    (framework / ".env").write_text("\n".join([
        "SCHEME=http",
        "HOST=gateway",
        "PORT=8001",
        "ARC_API_KEY=competition-gateway",
        "ARC_BASE_URL=http://gateway:8001/",
        "OPERATION_MODE=online",
        "ENVIRONMENTS_DIR=",
        f"RECORDINGS_DIR={WORKING}/recordings",
        f"LLM_BASE_URL={VLLM_BASE_URL}",
        f"LLM_MODEL_ID={SERVED_MODEL}",
        "LLM_API_KEY=EMPTY",
        "LLM_CONTEXT_WINDOW=32768",
        "LLM_MAX_OUTPUT=2048",
        "LLM_TEMPERATURE=0.6",
        "LLM_TIMEOUT=120",
        "LLM_MULTIMODAL=0",
    ]) + "\n")
    
    # Wait for gateway
    import urllib.request
    print("Waiting for competition gateway...")
    deadline = time.time() + 600
    while time.time() < deadline:
        try:
            with urllib.request.urlopen("http://gateway:8001/api/games", timeout=10) as r:
                if r.status < 500:
                    break
        except Exception:
            pass
        time.sleep(5)
    
    # Run the agent
    run_env = os.environ.copy()
    run_env["MPLBACKEND"] = "Agg"
    run_env["LLM_BASE_URL"] = VLLM_BASE_URL
    run_env["LLM_MODEL_ID"] = SERVED_MODEL
    run_env["PYTHONPATH"] = str(SITE_PACKAGES) + os.pathsep + run_env.get("PYTHONPATH", "")
    
    # Choose agent: LLM if vLLM started successfully, otherwise fallback to heuristic
    if VLLM_OK:
        agent_name = "llmagent"
        print(f"Starting LLM agent run ({agent_name})...")
    else:
        agent_name = "myagent"
        print("WARNING: vLLM not available, falling back to heuristic agent (myagent)")
    
    result = subprocess.run(
        [sys.executable, "main.py", "--agent", agent_name],
        cwd=framework,
        env=run_env,
        capture_output=True,
        text=True,
        timeout=32000,  # ~9 hours
    )
    print("STDOUT:", result.stdout[-3000:] if result.stdout else "(empty)")
    print("STDERR:", result.stderr[-3000:] if result.stderr else "(empty)")
    print(f"Return code: {result.returncode}")
    
else:
    # === SAVE PHASE (non-competition) ===
    import pandas as pd
    placeholder = pd.DataFrame(
        [["placeholder_0", "placeholder", True, 0.0]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    )
    placeholder.to_parquet(WORKING / "submission.parquet", index=False)
    print("Wrote save-phase placeholder submission.parquet")


In [ ]:
# Print diagnostics
elapsed = time.time() - NOTEBOOK_START
print(f"Notebook elapsed: {elapsed:.0f}s ({elapsed/60:.1f} min)")

# Show vLLM server log tail
if VLLM_LOG.exists():
    log = VLLM_LOG.read_text()
    print(f"\nvLLM log ({len(log)} chars), last 500:")
    print(log[-500:])

# Show any recordings
recordings = WORKING / "recordings"
if recordings.exists():
    files = list(recordings.iterdir())
    print(f"\nRecordings: {len(files)} files")
    for f in files[:5]:
        print(f"  {f.name} ({f.stat().st_size} bytes)")
